# The Plasm Koans

Each koan below has blanks (`__`) to fill in. Run the setup cell once, then run each koan cell:
it runs the koan in its own sandbox, so every koan starts from the same data, and shows ✔ when
it passes, or what to fill in or fix. Edit a cell and run it again until it passes.

Build the koans first: `dotnet build koans/Plasm.Koans`. The setup cell starts a throwaway
PostgreSQL container, or uses the server named by `PLASM_KOANS_PG`.

In [ ]:
#I "../Plasm.Koans/bin/Debug/net11.0"
#r "Microsoft.Extensions.Logging.Abstractions.dll"
#r "Plasm.Annotations.dll"
#r "Plasm.dll"
#r "Plasm.Postgres.dll"
#r "Npgsql.dll"
#r "Testcontainers.dll"
#r "Testcontainers.PostgreSql.dll"
#r "Plasm.Koans.Core.dll"
#r "Plasm.Koans.dll"
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop
open Plasm.Koans.Notebook

start ()

## Getting started

A koan is a few lines of F# with a blank (__) to fill in. This lesson shows how a blank works, and the F# the course relies on: values, lists and the pipe, records and with, wrappers, options, results and tasks.

In [ ]:
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koan "Fill in the blank" (fun _ ->
    // A koan is a few lines of F# with a blank, written __. Replace
    // the blank with a value and run the koans again.
    // `actual |> shouldBe expected` passes when the two are equal.
    let name = "Widget"
    name |> shouldBe __)

In [ ]:
koan "Values do not change" (fun _ ->
    // let names a value. The name keeps that value: working out a
    // new value gives a new name, and the old one is unchanged.
    let price = 10m
    let discounted = price - 2m
    discounted |> shouldBe __
    price |> shouldBe __)

In [ ]:
koan "The pipe passes a value along" (fun _ ->
    // [ ... ] is a list. The pipe, |>, hands the value on its left
    // to the function on its right: `xs |> List.sort` is `List.sort xs`.
    let totals = [ 30m; 12.5m; 99m ]
    totals |> List.sort |> List.head |> shouldBe __)

In [ ]:
koan "A record groups named fields" (fun _ ->
    // A record is a value with named fields. Product is a record
    // defined in Shop.fs, with an Id, a Sku and a Name.
    let widget = { Id = ProductId 1L; Sku = "WIDGET"; Name = "Widget" }
    widget.Sku |> shouldBe __)

In [ ]:
koan "with makes a changed copy" (fun _ ->
    // Records cannot be changed. `{ r with Field = value }` makes a
    // new record with that field changed and every other field copied.
    let widget = { Id = ProductId 1L; Sku = "WIDGET"; Name = "Widget" }
    let renamed = { widget with Name = "Blue widget" }
    renamed.Name |> shouldBe __
    widget.Name |> shouldBe __)

In [ ]:
koan "A wrapper keeps a value's meaning" (fun _ ->
    // `type CustomerId = CustomerId of int64` (in Shop.fs) wraps a
    // number, so a customer's id cannot be mixed up with an order's.
    // Matching on the case takes the number back out.
    let ada = CustomerId 1L
    let (CustomerId n) = ada
    n |> shouldBe __)

In [ ]:
koan "An option may be absent" (fun _ ->
    // An option is either Some value or None (nothing there).
    // Option.defaultValue gives the value, or the default for None.
    let reason: string option = Some "duplicate"
    let noReason: string option = None
    reason |> Option.defaultValue "no reason" |> shouldBe __
    noReason |> Option.defaultValue "no reason" |> shouldBe __)

In [ ]:
koan "A result is Ok or Error" (fun _ ->
    // A Result is either Ok value or Error reason. Quantity.create
    // (in Shop.fs) refuses a quantity that is not positive.
    // `ok r` takes the value out of Ok, `err r` the reason out of Error;
    // given the other case, they fail the koan.
    let three = Quantity.create 3
    let zero = Quantity.create 0
    (ok three).Value |> shouldBe __
    err zero |> shouldBe __)

In [ ]:
koanTask "let! waits for a task" (fun _ ->
    // A task is work that finishes later, such as a database call.
    // Inside task { ... }, let! waits for a task and names its result.
    task {
        let countLater () = task { return 2 + 2 }
        let! n = countLater ()
        n |> shouldBe __
    })

## What a repo is

Every koan is given repo, a Plasm repository over the koans database. A repo runs queries and writes against one database with one model. A query is only a description: the repo turns it into SQL, sends it when asked, and returns a Result.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koanTask "Every koan is given a repo" (fun repo ->
    task {
        // repo is a Repo (a repository): Plasm's way in to a database.
        // It runs queries and writes. Every koan is given one, and
        // whatever a koan changes is undone when it ends. Count asks how
        // many rows a query matches; Query.from (tableOf<Customer>) is every
        // row of the customers table.
        let! customers = repo.Count(Query.from (tableOf<Customer>))
        customers |> shouldBe (Ok __)
    })

In [ ]:
koanTask "ok opens a Result" (fun repo ->
    task {
        // A repo answers with a Result: Ok with the answer, or Error
        // saying what went wrong. ok (under Given) takes the value out
        // of an Ok; given an Error it fails the koan and shows the error.
        let! orders = repo.Count(Query.from (tableOf<Order>))
        orders |> shouldBe (Ok 4L)
        ok orders |> shouldBe __
    })

In [ ]:
koanTask "A repo pairs a model with a database" (fun repo ->
    task {
        // A repo (repository) is where Plasm meets the database. It is
        // made from a model, the record types Plasm can store (Db.model,
        // built from the records in Shop.fs), and a database connection.
        repo.Model.Types.Length |> shouldBe __

        // CheckSchema compares the model with the database's tables
        // and lists every difference it finds.
        let! issues = repo.CheckSchema()
        ok issues |> shouldBe __
    })

In [ ]:
koanTask "In your app, you make the repo" (fun repo ->
    task {
        // A Repo is made from a model, the record types it can store
        // (Db.model, from Shop.fs), and a provider, the connection to
        // the database. In your app the provider is
        // Postgres.connect connectionString; here it is the koans' own,
        // repo.Provider. Both repos read the same database.
        let mine = Repo(Db.model, repo.Provider)
        let! customers = mine.Count(Query.from (tableOf<Customer>))
        ok customers |> shouldBe __
    })

In [ ]:
koanTask "sqlSent () is the SQL a koan has sent" (fun repo ->
    task {
        // sqlSent () lists the SQL this koan has sent to the database
        // so far, in order; every koan starts with none. Counting it
        // shows when Plasm talks to the database. (In your app, Plasm
        // reports each command to RepoOptions.OnCommand instead.)
        sqlSent () |> shouldBe __
        let! _ = repo.Count(Query.from (tableOf<Customer>))
        sqlSent().Length |> shouldBe __
    })

In [ ]:
koan "A query is a description" (fun repo ->
    // A query is a value that describes what to read. Building it
    // sends nothing.
    let _customers = Query.from (tableOf<Customer>)
    sqlSent().Length |> shouldBe __)

In [ ]:
koan "The repo shows its SQL" (fun repo ->
    // A repo turns a query into SQL. repo.Explain shows its plan
    // without sending anything; the plan's Sql is the SQL it would send.
    let products = Query.from (tableOf<Product>)
    let sql = (ok (repo.Explain products)).Sql
    sql.Contains "FROM \"products\"" |> shouldBe __
    sqlSent().Length |> shouldBe __)

In [ ]:
koanTask "Running sends that SQL" (fun repo ->
    task {
        // The repo sends a query when you run it, with All, One or Count.
        let products = Query.from (tableOf<Product>)
        let sql = (ok (repo.Explain products)).Sql
        let! _ = repo.All products
        sqlSent().Length |> shouldBe __
        (sqlSent().Head = sql) |> shouldBe __
    })

In [ ]:
koanTask "A query runs each time you ask" (fun repo ->
    task {
        // A repo does not remember results: each run reads the
        // database again. The query itself is unchanged by running it.
        let products = Query.from (tableOf<Product>)
        let! first = repo.All products
        let! second = repo.All products
        sqlSent().Length |> shouldBe __
        (ok first = ok second) |> shouldBe __
    })

In [ ]:
koanTask "Every call returns a Result" (fun repo ->
    task {
        // Every repo call returns a Result: Ok with what it read, or
        // Error with what went wrong. One returns an option: the row,
        // or None. `plasm { ... where ... }` keeps only matching rows.
        let! orders = repo.Count(Query.from (tableOf<Order>))
        let! zed = repo.One(plasm { for c in tableOf<Customer> do where (c.Name = "Zed") })
        orders |> shouldBe __
        zed |> shouldBe __
    })

In [ ]:
koanTask "A repo knows only its model" (fun repo ->
    task {
        // A model that knows only products, and a repo over it on the
        // same database: Repo(model, provider), as in your app.
        let productsOnly = Model.buildOrFail [] [ typeof<Product> ]
        let! customers = Repo(productsOnly, repo.Provider).All(Query.from (tableOf<Customer>))

        // It cannot read customers: it returns an Error, it does not throw.
        match err customers with
        | DbError.Translation(TranslationError.UnmappedType name) -> name |> shouldBe __
        | other -> failwithf "expected an unmapped type, got %A" other

        sqlSent().Length |> shouldBe __
    })

## Meet the shop

The koans database is a small shop: customers place orders, each order holds its lines, and each line is for a product. Every koan starts from the same data, and its changes vanish when it ends.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koanTask "The shop has three customers" (fun repo ->
    task {
        // All reads every row a query matches, each as a record: here
        // every customer as a Customer (under Given).
        let! customers = repo.All(Query.from (tableOf<Customer>))
        let names = ok customers |> List.map (fun c -> c.Name) |> List.sort
        names |> shouldBe __
    })

In [ ]:
koanTask "Counting rows" (fun repo ->
    task {
        // Count asks the database how many rows there are, without
        // reading them. The starting data has three products.
        let! products = repo.Count(Query.from (tableOf<Product>))
        ok products |> shouldBe __
    })

In [ ]:
koanTask "The database numbers the rows" (fun repo ->
    task {
        // Every table's id is generated by the database: 1, 2, 3 ...
        // in the order the starting data inserts the rows. The id
        // comes back wrapped in its own type (CustomerId, in Shop.fs).
        let! customers = repo.All(Query.from (tableOf<Customer>))
        let bob = ok customers |> List.find (fun c -> c.Name = "Bob")
        bob.Id |> shouldBe __
    })

In [ ]:
koanTask "An order belongs to a customer" (fun repo ->
    task {
        // An order's CustomerId says who placed it: the
        // customer_id column of the orders table.
        let! customers = repo.All(Query.from (tableOf<Customer>))
        let! orders = repo.All(Query.from (tableOf<Order>))
        let ada = ok customers |> List.find (fun c -> c.Name = "Ada")
        let adas = ok orders |> List.filter (fun o -> o.CustomerId = Some ada.Id)
        adas.Length |> shouldBe __
    })

In [ ]:
koanTask "An order may have no customer" (fun repo ->
    task {
        // customer_id may be null, so CustomerId is an option.
        let! orders = repo.All(Query.from (tableOf<Order>))
        let fourth = ok orders |> List.find (fun o -> o.Id = OrderId 4L)
        fourth.CustomerId |> shouldBe __
    })

In [ ]:
koanTask "An order holds its lines" (fun repo ->
    task {
        // An Order holds its OrderLines as a plain list: its own rows
        // in the order_lines table, read with the order every time.
        let! orders = repo.All(Query.from (tableOf<Order>))
        let first = ok orders |> List.find (fun o -> o.Id = OrderId 1L)
        first.Lines |> List.map (fun l -> l.ProductId) |> shouldBe __
    })

In [ ]:
koanTask "A koan's changes stay in the koan" (fun repo ->
    task {
        // Every koan runs in a sandbox: a transaction that is rolled
        // back when the koan ends. Inside it, changes are real.
        // (repo.Transact runs a transaction, and Tx.insert is a step
        // that inserts a record; lessons 11 and 13 explain them.)
        let addDan: Tx<Customer, unit> = Tx.insert (New.customer "Dan" "dan@example.com")
        let! _ = repo.Transact addDan
        let! customers = repo.Count(Query.from (tableOf<Customer>))
        ok customers |> shouldBe __
    })

In [ ]:
koanTask "The next koan starts afresh" (fun repo ->
    task {
        // The previous koan inserted Dan. This koan has its own sandbox.
        let! customers = repo.Count(Query.from (tableOf<Customer>))
        ok customers |> shouldBe __
    })

## Plasm's promises

Plasm keeps a few promises everywhere: plain records, nothing run until asked, no hidden or in-memory work, failures as values, minimal writes, a checked model, F# meaning, batched loads and tests that roll back. Each koan shows one and names the lesson that goes deeper.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

/// A record with a field Plasm cannot store: a Uri has no column type, and it
/// is not a wrapper Plasm knows how to store by convention.
[<Table "bookmarks">]
type Bookmark = { Id: int64; Address: System.Uri }

In [ ]:
koanTask "Records stay plain F#" (fun repo ->
    task {
        // A record read from the database is an ordinary F# value: it
        // is not watched, and changing a copy writes nothing.
        // (Lesson 05 goes deeper.)
        let! products = repo.All(Query.from (tableOf<Product>))
        let widget = ok products |> List.find (fun p -> p.Sku = "WIDGET")
        let _renamed = { widget with Name = "Blue widget" }
        widget.Name |> shouldBe __
        sqlSent().Length |> shouldBe __
    })

In [ ]:
koan "Nothing runs until you ask" (fun repo ->
    // A query and a transaction (a description of writes) are values.
    // Only a repo call sends anything. (Lessons 06 and 13 go deeper.)
    let _drafts = plasm { for o in tableOf<Order> do where (o.Status = Draft) }
    let _addDan: Tx<Customer, unit> = Tx.insert (New.customer "Dan" "dan@example.com")
    sqlSent().Length |> shouldBe __)

In [ ]:
koanTask "No hidden queries" (fun repo ->
    task {
        // An order's Lines are read with the order, every time: one
        // command for the order, one for its lines. Reading o.Lines
        // afterwards is reading a list; it never queries the database
        // behind your back. (Lesson 09 goes deeper.)
        let! first = repo.One(plasm { for o in tableOf<Order> do where (o.Id = OrderId 1L) })
        let first = (ok first).Value
        first.Lines.Length |> shouldBe __
        first.Lines |> List.map (fun l -> l.UnitPrice) |> ignore
        sqlSent().Length |> shouldBe __
    })

In [ ]:
koanTask "Nothing is evaluated behind your back" (fun repo ->
    task {
        // shout is ordinary F#, which SQL cannot run. Plasm refuses the
        // query rather than reading every customer to run it in memory.
        // (Lesson 07 goes deeper.)
        let shout (s: string) = s.ToUpperInvariant() + "!"
        let! r = repo.All(plasm { for c in tableOf<Customer> do where (shout c.Name = "ADA!") })
        Result.isError r |> shouldBe __
        sqlSent().Length |> shouldBe __
    })

In [ ]:
koanTask "A failure is a value" (fun repo ->
    task {
        // Ada's email is taken, and tableOf<Customer> states the email is
        // unique. The failure comes back as an Error you can match on,
        // here as invalid changes naming the field; nothing is thrown.
        // (Lessons 10 and 20 go deeper.)
        let addAdaAgain: Tx<Customer, unit> = Tx.insert (New.customer "Ada again" "ada@example.com")
        let! r = repo.Transact addAdaAgain

        match (err r).Failure with
        | TxFailure.InvalidChanges(_, errors) -> errors |> List.map (fun e -> e.Field) |> shouldBe __
        | other -> failwithf "expected invalid changes, got %A" other
    })

In [ ]:
koanTask "Only what you change is written" (fun repo ->
    task {
        // Tx.update (original, changed) describes a change to a record you
        // read. The UPDATE sets only the fields that differ. (Lesson 11
        // goes deeper.)
        let! drafts = repo.All(plasm { for o in tableOf<Order> do where (o.Status = Draft) })
        let draft = ok drafts |> List.head
        let publish: Tx<Order, unit> = Tx.update (draft, { draft with Status = Published })
        let! _ = repo.Transact publish
        let update = List.last (sqlSent ())
        update.Contains "\"status\" = " |> shouldBe __
        update.Contains "\"total\" = " |> shouldBe __
    })

In [ ]:
koan "The model is checked before any query" (fun repo ->
    // Model.build checks every record type once, when the model is made.
    // It cannot store a Bookmark's Address (above, under Given).
    // Each error names the record and the field (Property) at fault.
    // (Lesson 05 goes deeper.)
    match Model.build [] [ typeof<Bookmark> ] with
    | Ok _ -> failwith "expected a configuration error"
    | Error errors ->
        (errors |> List.map (fun e -> e.Property) |> List.distinct) |> shouldBe __
        sqlSent().Length |> shouldBe __)

In [ ]:
koanTask "Queries mean what F# means" (fun repo ->
    task {
        // In F#, `o.CustomerId = None` is true for an order with no
        // customer. In the query it means the same: SQL's IS NULL.
        // (Lesson 07 goes deeper.)
        let noCustomer = plasm { for o in tableOf<Order> do where (o.CustomerId = None) }
        (ok (repo.Explain noCustomer)).Sql.Contains "IS NULL" |> shouldBe __
        let! n = repo.Count noCustomer
        ok n |> shouldBe __
    })

In [ ]:
koanTask "What an order owns loads in one batch" (fun repo ->
    task {
        // Reading all four orders reads all their lines too: in one
        // more command, not one command per order. (Lesson 09 goes deeper.)
        let! _ = repo.All(Query.from (tableOf<Order>))
        sqlSent().Length |> shouldBe __
    })

In [ ]:
koanTask "Tests roll back" (fun repo ->
    task {
        // repo.Sandbox runs code in a transaction that is rolled back
        // when it ends: every koan runs in one. (Lesson 21 goes deeper.)
        let! inside =
            repo.Sandbox(fun repo ->
                task {
                    let addDan: Tx<Customer, unit> = Tx.insert (New.customer "Dan" "dan@example.com")
                    let! _ = repo.Transact addDan
                    return! repo.Count(Query.from (tableOf<Customer>))
                })

        let! after = repo.Count(Query.from (tableOf<Customer>))
        ok inside |> shouldBe __
        ok after |> shouldBe __
    })

## About stored records

Domain types are ordinary immutable F# records and unions. A stored record names its table with [<Table>], and everything else follows from its types: its key is its field named Id, a column is named after its field, and ids, validated values and fieldless unions are stored by convention, with no mappings. A model is built once from the record types, before any query runs, and refuses what the types leave unclear.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

/// A record with two fields Plasm cannot store: a Uri has no column type, and
/// it is not a wrapper or a union Plasm can store by convention.
[<Table "visits">]
type Visit =
    { Id: int64
      Page: System.Uri
      Referrer: System.Uri }

/// A wish: a plain record that names no table, so it is not stored.
type Wish = { Item: string }

/// A wishlist holding wishes. A list is stored only if its records are.
[<Table "wishlists">]
type Wishlist = { Id: int64; Wishes: Wish list }

/// A depot's place: a record that names no table and has no Id. As one field
/// of a stored record it is a value, kept in its owner's row.
type Place = { Street: string; City: string }

[<Table "depots">]
type Depot = { Id: int64; Name: string; Place: Place }

/// A chore's steps: kept as one JSON document in the chore's row.
type Step = { Text: string; Done: bool }

[<Table "chores">]
type Chore =
    { Id: int64
      Title: string
      [<Json>]
      Steps: Step list }

In [ ]:
koan "A stored record is still plain F#" (fun _ ->
    // Product is an ordinary immutable F# record. In Shop.fs it carries
    // [<Table "products">], a note saying where it is stored; it has no
    // base class and nothing to call. Copy-and-update works as always.
    let widget = { Id = ProductId 1L; Sku = "WIDGET"; Name = "Widget" }
    let renamed = { widget with Name = "Better widget" }
    widget.Name |> shouldBe __
    renamed.Name |> shouldBe __)

In [ ]:
koan "A record names its table" (fun repo ->
    // [<Table "products">] on Product names the table it is stored in.
    // tableOf<Product> is every stored Product, as a query source, and
    // Query.from makes a query for all of them.
    // repo.Explain shows the SQL a query would send, without sending it.
    let sql = (ok (repo.Explain(Query.from (tableOf<Product>)))).Sql
    // Which table does the query read from?
    sql.Contains "FROM \"products\"" |> shouldBe __)

In [ ]:
koan "Columns are named in snake_case" (fun repo ->
    // A column is named after its field, in snake_case: lower case words
    // joined by underscores. Nothing on Order says so; it is the rule.
    let sql = (ok (repo.Explain(Query.from (tableOf<Order>)))).Sql
    // Order has a field PlacedAt. Which name does the SQL use?
    sql.Contains "\"placed_at\"" |> shouldBe __
    sql.Contains "\"PlacedAt\"" |> shouldBe __)

In [ ]:
koanTask "A key finds one record" (fun repo ->
    task {
        // A stored record's key is its field named Id. A key that wraps a
        // number, such as ProductId, is numbered by the database: 1, 2, 3, in
        // the order the rows were inserted.
        // repo runs queries against the database; One reads one record, if any.
        let! product = repo.One(plasm { for p in tableOf<Product> do where (p.Id = ProductId 2L) })
        // ok takes the value out of an Ok result; .Value takes it out of Some.
        (ok product).Value.Name |> shouldBe __
    })

In [ ]:
koanTask "A union is stored as text" (fun repo ->
    task {
        // OrderStatus is a union with no fields. Plasm stores such a union
        // by convention, each case as its name in snake_case: Draft as
        // 'draft', Published as 'published', and so on. Nothing registers it.
        // Here the status column of Order 2 is changed with SQL, as text.
        // (Tx.executeSql is a step running SQL, in a transaction that
        // repo.Transact runs; lesson 18 goes deeper. The change is made
        // in this koan's sandbox and vanishes when it ends.)
        let cancel: Tx<int, unit> = Tx.executeSql $"update orders set status = 'cancelled', cancellation_reason = 'test' where id = 2"
        let! _ = repo.Transact cancel
        let! order = repo.One(plasm { for o in tableOf<Order> do where (o.Id = OrderId 2L) })
        // Is Status the text, or a case of OrderStatus?
        (ok order).Value.Status |> shouldBe __
    })

In [ ]:
koanTask "Wrapped keys keep their meaning" (fun repo ->
    task {
        // CustomerId is a single-case union over int64. By convention Plasm
        // stores it as the number it wraps, and reads it back wrapped.
        // An Email is stored the same way: email (in Shop.fs) makes one.
        let adaEmail = email "ada@example.com"
        let! ada = repo.One(plasm { for c in tableOf<Customer> do where (c.Email = adaEmail) })
        let ada = (ok ada).Value
        ada.Id |> shouldBe __
    })

In [ ]:
koanTask "Stored values can be rejected" (fun repo ->
    task {
        // Quantity can only be built by Quantity.create, which refuses zero.
        Quantity.create 0 |> Result.isOk |> shouldBe __

        // Plasm reads a Quantity back through Quantity.create, so a zero
        // stored in the database is refused on the way in, with its field named.
        let zero: Tx<int, unit> = Tx.executeSql $"update order_lines set quantity = 0 where id = 1"
        let! _ = repo.Transact zero
        let! lines = repo.All(Query.from (tableOf<OrderLine>))

        match lines with
        | Error(DbError.Materialization m) -> m.Property |> shouldBe __
        | other -> failwithf "expected a materialisation error, got %A" other
    })

In [ ]:
koanTask "Absent values are options" (fun repo ->
    task {
        // Order 4 has no customer (a NULL column) but has a cancellation reason.
        let! order = repo.One(plasm { for o in tableOf<Order> do where (o.Id = OrderId 4L) })
        let order = (ok order).Value
        order.CustomerId |> shouldBe __
        order.CancellationReason |> shouldBe __
    })

In [ ]:
koan "A model reports every mistake at once" (fun _ ->
    // A Visit (above, under Given) has two fields Plasm cannot store.
    // Model.build makes a model from the record types it lists, and
    // reports every problem, not just the first.
    match Model.build [] [ typeof<Visit> ] with
    | Ok _ -> failwith "expected configuration errors"
    | Error errors ->
        errors.Length |> shouldBe __
        errors |> List.map (fun e -> e.Property) |> shouldBe __)

In [ ]:
koan "A list is stored only if its records are" (fun _ ->
    // A Wishlist (under Given) holds a list of Wish, a record that names
    // no table. Plasm does not guess where the wishes would go: building
    // the model refuses it, naming the field.
    match Model.build [] [ typeof<Wishlist> ] with
    | Ok _ -> failwith "expected a configuration error"
    | Error errors -> errors |> List.map (fun e -> e.Property) |> shouldBe __)

In [ ]:
koan "One record without a table lives in its owner's row" (fun repo ->
    // Place (under Given) names no table, but a Depot holds one Place,
    // not a list: it is stored in the depots row, each of its fields in
    // a column named <field>_<subfield>. repo.Ddl prints the tables a
    // model expects, without touching the database.
    let ddl = Repo(Model.buildOrFail [] [ typeof<Depot> ], repo.Provider).Ddl()
    ddl.Contains "place_city text not null" |> shouldBe __)

In [ ]:
koan "A field marked Json is one document" (fun repo ->
    // Chore (under Given) marks its Steps [<Json>]: the whole list is kept
    // as one jsonb document in the chores row, each step an object keyed
    // in snake case, {"text": ..., "done": ...}. Queries can still reach
    // into it: List.exists (fun s -> not s.Done) c.Steps runs in SQL.
    let ddl = Repo(Model.buildOrFail [] [ typeof<Chore> ], repo.Provider).Ddl()
    ddl.Contains "steps jsonb not null" |> shouldBe __)

## About query values

A query is a value: building, extending and inspecting it never touches the database. Execution happens only at the repository.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koanTask "Query.from reads every row" (fun repo ->
    task {
        // A query describes which rows to read. Query.from is the plainest
        // query there is: every row of a record's table, as it is.
        // repo.All runs a query and returns all the rows it matches.
        let! customers = repo.All(Query.from (tableOf<Customer>))
        // How many customers does the koans database start with?
        (ok customers).Length |> shouldBe __
    })

In [ ]:
koanTask "A where keeps the matching rows" (fun repo ->
    task {
        // plasm { } writes a query step by step. `for o in tableOf<Order> do` names
        // each Order o; `where` keeps the rows for which its condition is true.
        let drafts = plasm { for o in tableOf<Order> do where (o.Status = Draft) }
        let! orders = repo.All drafts
        // How many Orders are drafts?
        (ok orders).Length |> shouldBe __
    })

In [ ]:
koanTask "An orderBy sorts the rows" (fun repo ->
    task {
        // orderBy sorts the rows by a field, smallest first (for text: A to Z).
        // The database does the sorting.
        let byName = plasm { for p in tableOf<Product> do orderBy p.Name }
        let! products = repo.All byName
        ok products |> List.map (fun p -> p.Name) |> shouldBe __
    })

In [ ]:
koanTask "take keeps the first rows" (fun repo ->
    task {
        // take n keeps the first n rows. "First" needs an order, so the
        // query sorts the Customers by name before taking two.
        let firstTwo = plasm { for c in tableOf<Customer> do orderBy c.Name; take 2 }
        let! customers = repo.All firstTwo
        ok customers |> List.map (fun c -> c.Name) |> shouldBe __
    })

In [ ]:
koan "Building a query runs nothing" (fun repo ->
    let _published = plasm { for o in tableOf<Order> do where (o.Status = Published) }
    // How many commands has this koan sent?
    sqlSent().Length |> shouldBe __)

In [ ]:
koanTask "Extending makes a new query" (fun repo ->
    task {
        let all = Query.from (tableOf<Order>)
        let published = plasm { for o in all do where (o.Status = Published) }
        let! everyOrder = repo.Count all
        let! publishedOrders = repo.Count published
        ok everyOrder |> shouldBe __
        ok publishedOrders |> shouldBe __
    })

In [ ]:
koanTask "Running does not change a query" (fun repo ->
    task {
        let drafts = plasm { for o in tableOf<Order> do where (o.Status = Draft) }
        let! first = repo.All drafts
        let! second = repo.All drafts
        (ok first = ok second) |> shouldBe __
        (ok first).Length |> shouldBe __
    })

In [ ]:
koan "A query can explain itself" (fun repo ->
    let published = plasm { for o in tableOf<Order> do where (o.Status = Published) }
    let sql = (ok (repo.Explain published)).Sql
    // Is the status sent as a parameter?
    sql.Contains "\"status\" = @p0" |> shouldBe __
    sqlSent().Length |> shouldBe __)

In [ ]:
koanTask "Reusable filters are functions" (fun repo ->
    task {
        let published (q: Query<Order>) = plasm { for o in q do where (o.Status = Published) }
        let over (amount: decimal) (q: Query<Order>) = plasm { for o in q do where (o.Total > amount) }
        let! n = repo.Count(Query.from (tableOf<Order>) |> published |> over 50m)
        ok n |> shouldBe __
    })

In [ ]:
koanTask "Captured values become parameters" (fun repo ->
    task {
        let bobs = email "bob@example.com"
        let byEmail = plasm { for c in tableOf<Customer> do where (c.Email = bobs) }
        // Is the email written into the SQL text?
        (ok (repo.Explain byEmail)).Sql.Contains "bob@example.com" |> shouldBe __
        let! bob = repo.One byEmail
        (ok bob).Value.Name |> shouldBe __
    })

## About filters and ordering

Filters mean what the same F# predicate means. Anything that cannot run in the database is refused, never evaluated in memory.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let count (repo: Repo) (q: Query<Order>) =
    task {
        let! n = repo.Count q
        return ok n
    }

In [ ]:
koanTask "Comparing numbers" (fun repo ->
    task {
        // A filter compares a field as F# would: Total is a decimal, so it is
        // compared with a decimal (20m). count, defined at the top of this
        // lesson, runs repo.Count: how many rows the query matches.
        let! n = count repo (plasm { for o in tableOf<Order> do where (o.Total > 20m) })
        n |> shouldBe __
    })

In [ ]:
koanTask "Contains on text becomes LIKE" (fun repo ->
    task {
        // s.Contains "o" is true when the text s has an "o" in it. In a filter it
        // becomes SQL's LIKE, which matches text against a pattern.
        let withO = plasm { for c in tableOf<Customer> do where (c.Name.Contains "o") }
        (ok (repo.Explain withO)).Sql.Contains "LIKE" |> shouldBe __
        let! customers = repo.All withO
        ok customers |> List.map (fun c -> c.Name) |> shouldBe __
    })

In [ ]:
koanTask "And needs both" (fun repo ->
    task {
        // && keeps a row only when both sides are true.
        let! n = count repo (plasm { for o in tableOf<Order> do where (o.Status = Published && o.Total < 50m) })
        n |> shouldBe __
    })

In [ ]:
koanTask "Or needs either" (fun repo ->
    task {
        // || keeps a row when either side is true, or both.
        let! n = count repo (plasm { for o in tableOf<Order> do where (o.Status = Draft || o.Total > 50m) })
        n |> shouldBe __
    })

In [ ]:
koanTask "Ordering largest first" (fun repo ->
    task {
        // orderByDescending sorts the rows largest first.
        let! orders = repo.All(plasm { for o in tableOf<Order> do orderByDescending o.Total })
        ok orders |> List.map (fun o -> o.Total) |> shouldBe __
    })

In [ ]:
koanTask "Independent filters combine" (fun repo ->
    task {
        let! n =
            count repo (plasm {
                for o in tableOf<Order> do
                    where (o.Status = Published)
                    where (o.Total > 50m)
            })

        n |> shouldBe __
    })

In [ ]:
koanTask "Comparing with None" (fun repo ->
    task {
        let withoutCustomer = plasm { for o in tableOf<Order> do where (o.CustomerId = None) }
        (ok (repo.Explain withoutCustomer)).Sql.Contains "IS NULL" |> shouldBe __
        let! n = count repo withoutCustomer
        n |> shouldBe __
    })

In [ ]:
koanTask "not keeps F#'s meaning" (fun repo ->
    task {
        // Orders that are not Ada's: in F#, an Order with no customer is not Ada's.
        let notAdas = plasm { for o in tableOf<Order> do where (not (o.CustomerId = Some(CustomerId 1L))) }
        let! n = count repo notAdas
        n |> shouldBe __
    })

In [ ]:
koanTask "Membership in an empty list" (fun repo ->
    task {
        let wanted: OrderStatus list = []
        let! n = count repo (plasm { for o in tableOf<Order> do where (List.contains o.Status wanted) })
        n |> shouldBe __
    })

In [ ]:
koanTask "Some code cannot be translated" (fun repo ->
    task {
        let shout (s: string) = s.ToUpperInvariant() + "!"
        let! r = repo.All(plasm { for c in tableOf<Customer> do where (shout c.Name = "ADA!") })

        let refused =
            match r with
            | Error(DbError.Translation(TranslationError.UnsupportedExpression _)) -> true
            | _ -> false

        refused |> shouldBe __
    })

In [ ]:
koanTask "Pages need an order" (fun repo ->
    task {
        let! unordered = repo.All(plasm { for o in tableOf<Order> do take 2 })
        Result.isError unordered |> shouldBe __

        let! page = repo.All(plasm { for o in tableOf<Order> do orderBy o.Total; take 2 })
        ok page |> List.map (fun o -> o.Total) |> shouldBe __
    })

In [ ]:
koanTask "Captured arrays are read each run" (fun repo ->
    task {
        let statuses = [| Published |]
        let matching = plasm { for o in tableOf<Order> do where (Array.contains o.Status statuses) }
        let! before = count repo matching
        statuses.[0] <- Draft
        let! after = count repo matching
        before |> shouldBe __
        after |> shouldBe __
    })

## About reading

The repository runs queries and returns Results. Counting and existence transfer no rows; projections read only the columns they use.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koanTask "All returns a list" (fun repo ->
    task {
        // repo.All returns a Result: Ok with an F# list of the matching
        // records, or Error saying what went wrong. ok takes the list out.
        let! products = repo.All(plasm { for p in tableOf<Product> do orderBy p.Id })
        let products: Product list = ok products
        products |> List.map (fun p -> p.Sku) |> shouldBe __
    })

In [ ]:
koanTask "One returns an option" (fun repo ->
    task {
        // repo.One reads at most one record. Its value is an option:
        // Some record when a row matches, None when none does.
        let! cy = repo.One(plasm { for c in tableOf<Customer> do where (c.Name = "Cy") })
        ok cy |> Option.map (fun c -> c.Name) |> shouldBe __
    })

In [ ]:
koanTask "Count returns an int64" (fun repo ->
    task {
        // repo.Count returns how many rows match, as an int64
        // (a 64-bit integer, written with an L: 7L).
        let! n = repo.Count(Query.from (tableOf<Product>))
        ok n |> shouldBe __
    })

In [ ]:
koanTask "Selecting one field" (fun repo ->
    task {
        // select says what to read for each row: here only the Name,
        // so the result is a list of strings, not of Customers.
        let! names = repo.All(plasm { for c in tableOf<Customer> do orderBy c.Id; select c.Name })
        ok names |> shouldBe __
    })

In [ ]:
koanTask "Reading returns typed rows" (fun repo ->
    task {
        let! orders = repo.All(plasm { for o in tableOf<Order> do orderBy o.Id })
        let first = (ok orders).Head
        // Total is a decimal; Status is an OrderStatus, not a string.
        first.Total |> shouldBe __
        first.Status |> shouldBe __
    })

In [ ]:
koanTask "Counting transfers no rows" (fun repo ->
    task {
        let! n = repo.Count(plasm { for o in tableOf<Order> do where (o.Status = Published) })
        ok n |> shouldBe __
        sqlSent().Head.Contains "COUNT(*)" |> shouldBe __
    })

In [ ]:
koanTask "Asking whether anything exists" (fun repo ->
    task {
        let! any = repo.Exists(plasm { for o in tableOf<Order> do where (o.Total > 1000m) })
        ok any |> shouldBe __
    })

In [ ]:
koanTask "Selecting only what you need" (fun repo ->
    task {
        let totals = plasm { for o in tableOf<Order> do orderBy o.Id; select (o.Id, o.Total) }
        let! rows = repo.All totals
        (ok rows).Head |> shouldBe __
        // Is the status column read?
        sqlSent().Head.Contains "\"status\"" |> shouldBe __
    })

In [ ]:
koanTask "Leaving the database is explicit" (fun repo ->
    task {
        let! succeeded = repo.All(plasm { for p in tableOf<Payment> do where (p.Status = Succeeded) })
        // From here on it is an ordinary F# list, in memory.
        ok succeeded |> List.sumBy (fun p -> p.Amount) |> shouldBe __
        sqlSent().Length |> shouldBe __
    })

In [ ]:
koanTask "One means at most one" (fun repo ->
    task {
        let! largest = repo.One(plasm { for o in tableOf<Order> do orderByDescending o.Total; take 1 })
        (ok largest |> Option.map (fun o -> o.Total)) |> shouldBe __
        let! none = repo.One(plasm { for o in tableOf<Order> do where (o.Total > 1000m) })
        ok none |> shouldBe __
    })

## About aggregates and references

An aggregate is a record with the records it owns, read whole: an Order is always read with its lines, in one more command for any number of orders. Other aggregates are referred to by id and reached with a query, a join, an exists, or including.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let byId (orders: Order list) (id: int64) = orders |> List.find (fun o -> o.Id = OrderId id)

/// An order's payments: the tie between the two aggregates, written once.
/// [<ReflectedDefinition>] lets Plasm read the function as part of a query.
[<ReflectedDefinition>]
let paymentsOf (o: Order) =
    plasm {
        for p in tableOf<Payment> do
            where (p.OrderId = o.Id)
    }

In [ ]:
koanTask "An order is read with its lines" (fun repo ->
    task {
        // An aggregate is a record together with the records it owns. An
        // Order owns its OrderLines: Order has a field Lines: OrderLine list,
        // and an OrderLine holds its order's key (OrderId). Whenever an
        // order is read, its Lines are read with it.
        // byId, at the top of this lesson, finds the Order with an id in a list.
        let! orders = repo.All(Query.from (tableOf<Order>))
        let order = byId (ok orders) 1L
        order.Lines.Length |> shouldBe __
    })

In [ ]:
koanTask "Lines come in the order of their key" (fun repo ->
    task {
        // Owned records are read in the order of their key, here l.Id, so
        // the order they were written in. [<OrderedBy>] on the field would
        // name another order.
        let! orders = repo.All(Query.from (tableOf<Order>))
        (byId (ok orders) 1L).Lines |> List.map (fun l -> l.Id) |> shouldBe __
    })

In [ ]:
koanTask "No lines is an empty list" (fun repo ->
    task {
        // Order 4 has no lines. An owned list is always read, so an empty
        // list means there are none: never "not read yet".
        let! orders = repo.All(Query.from (tableOf<Order>))
        (byId (ok orders) 4L).Lines |> shouldBe __
    })

In [ ]:
koanTask "Every order's lines come in one more command" (fun repo ->
    task {
        // Four orders: one command reads them, and one more reads the lines
        // of all four. Reading o.Lines afterwards never queries.
        let! orders = repo.All(Query.from (tableOf<Order>))

        for o in ok orders do
            o.Lines |> ignore

        sqlSent().Length |> shouldBe __
    })

In [ ]:
koanTask "A page of orders reads only its lines" (fun repo ->
    task {
        // take 2 keeps the first two orders. Their lines are read with them,
        // and only theirs: the second command asks for the lines of two orders.
        let! page = repo.All(plasm { for o in tableOf<Order> do orderBy o.Id; take 2 })
        ok page |> List.collect (fun o -> o.Lines) |> List.length |> shouldBe __
        sqlSent().Length |> shouldBe __
    })

In [ ]:
koanTask "A customer is referred to by id" (fun repo ->
    task {
        // A Customer is an aggregate of its own, so an Order holds only its
        // CustomerId. Because CustomerId is the key type of tableOf<Customer>,
        // Plasm knows the field refers to a customer.
        // To reach the customer, ask for it with a query.
        let! orders = repo.All(Query.from (tableOf<Order>))
        let customerId = (byId (ok orders) 3L).CustomerId
        customerId |> shouldBe __
        let id = customerId.Value
        let! customer = repo.One(plasm { for c in tableOf<Customer> do where (c.Id = id) })
        (ok customer).Value.Name |> shouldBe __
    })

In [ ]:
koanTask "A join reads both aggregates' fields" (fun repo ->
    task {
        // A join pairs each order with its customer in one command, and
        // select picks the fields to read. Order 4 has no customer, so an
        // inner join leaves it out. (Lesson 14 goes deeper.)
        let! rows =
            repo.All(
                plasm {
                    for o in tableOf<Order> do
                        join c in tableOf<Customer> on (o.CustomerId = Some c.Id)
                        orderBy o.Id
                        select (o.Id, c.Name)
                }
            )

        ok rows |> shouldBe __
    })

In [ ]:
koanTask "A whole order from a join comes with its lines" (fun repo ->
    task {
        // A whole Order is always read with its lines. Selected from a join,
        // the joined rows come from one command, and the lines of every
        // Order in them from one more.
        let! rows =
            repo.All(
                plasm {
                    for o in tableOf<Order> do
                        join c in tableOf<Customer> on (o.CustomerId = Some c.Id)
                        where (c.Name = "Ada")
                        orderBy o.Id
                        select (o, c.Name)
                }
            )

        let first, _ = List.head (ok rows)
        first.Lines.Length |> shouldBe __
        sqlSent().Length |> shouldBe __
    })

In [ ]:
koanTask "Filtering orders by their lines" (fun repo ->
    task {
        // List.exists over an owned list is a filter the database runs:
        // the orders with a line for product 2 (the Gadget), each once.
        let gadgets =
            plasm {
                for o in tableOf<Order> do
                    where (o.Lines |> List.exists (fun l -> l.ProductId = ProductId 2L))
                    orderBy o.Id
            }

        let! rows = repo.All gadgets
        ok rows |> List.map (fun o -> o.Id) |> shouldBe __
    })

In [ ]:
koanTask "Every line passes, as F# reads it" (fun repo ->
    task {
        // List.forall keeps the orders whose every line passes. As in F#,
        // an order with no lines passes: there is no line that fails.
        let tenOrMore =
            plasm {
                for o in tableOf<Order> do
                    where (o.Lines |> List.forall (fun l -> l.UnitPrice >= 10m))
                    orderBy o.Id
            }

        let! rows = repo.All tenOrMore
        ok rows |> List.map (fun o -> o.Id) |> shouldBe __
    })

In [ ]:
koanTask "Orders with a succeeded payment" (fun repo ->
    task {
        // A Payment is another aggregate: it refers to its order by OrderId.
        // exists (an inner query) inside a where asks whether any payment
        // matches this order, in the same command.
        let paid =
            plasm {
                for o in tableOf<Order> do
                    where (exists (plasm { for p in tableOf<Payment> do where (p.OrderId = o.Id && p.Status = Succeeded) }))
                    orderBy o.Id
            }

        let! rows = repo.All paid
        ok rows |> List.map (fun o -> o.Id) |> shouldBe __
    })

In [ ]:
koanTask "Orders without a succeeded payment" (fun repo ->
    task {
        // not (exists ...) keeps the orders no succeeded payment refers to.
        let unpaid =
            plasm {
                for o in tableOf<Order> do
                    where (not (exists (plasm { for p in tableOf<Payment> do where (p.OrderId = o.Id && p.Status = Succeeded) })))
                    orderBy o.Id
            }

        let! rows = repo.All unpaid
        ok rows |> List.map (fun o -> o.Id) |> shouldBe __
    })

In [ ]:
koanTask "Each order with its payments" (fun repo ->
    task {
        // A query ending in including reads each order with the list an
        // inner query finds for it. The inner query names the tie,
        // p.OrderId = o.Id; the result's type, (Order * Payment list),
        // says what was read. Plasm reads the payments of every order in
        // one more command and pairs them up.
        let withPayments =
            plasm {
                for o in tableOf<Order> do
                    orderBy o.Id
                    including (plasm { for p in tableOf<Payment> do where (p.OrderId = o.Id); orderBy p.Id })
            }

        let! rows = repo.All withPayments
        ok rows |> List.map (fun (o, payments) -> o.Id, payments.Length) |> shouldBe __
        sqlSent().Length |> shouldBe __
    })

In [ ]:
koanTask "A tie written once serves exists and including" (fun repo ->
    task {
        // paymentsOf (top of this lesson) is the tie, written once. It
        // works inside exists, and as the start of an inner query that
        // narrows it further. Inside including, orderBy and take apply to
        // each order's payments, and select picks what the list holds:
        // here, the status of the newest one of each.
        let paidOrders =
            plasm {
                for o in tableOf<Order> do
                    where (exists (paymentsOf o))
                    orderBy o.Id
                    including (plasm { for p in paymentsOf o do orderByDescending p.CreatedAt; take 1; select p.Status })
            }

        let! rows = repo.All paidOrders
        ok rows |> List.map (fun (o, newest) -> o.Id, newest) |> shouldBe __
    })

In [ ]:
koanTask "Each order with its payments and its deliveries" (fun repo ->
    task {
        // A query may end in up to eight includings. Each adds one list
        // to the tuple, in the order written, and is one more command:
        // two lists never multiply each other's rows.
        let withBoth =
            plasm {
                for o in tableOf<Order> do
                    orderBy o.Id
                    including (paymentsOf o)
                    including (plasm { for d in tableOf<Delivery> do where (d.OrderId = o.Id) })
            }

        let! rows = repo.All withBoth
        ok rows |> List.map (fun (o, payments, deliveries) -> o.Id, payments.Length, deliveries |> List.map (fun d -> d.StageName)) |> shouldBe __
        sqlSent().Length |> shouldBe __
    })

In [ ]:
koanTask "Each customer with their orders, each with its payments" (fun repo ->
    task {
        // An inner query may end in its own including, so its items come
        // with their own lists: Customer * (Order * Payment list) list.
        // An order's CustomerId is optional, so the tie compares it with
        // Some c.Id.
        let customers =
            plasm {
                for c in tableOf<Customer> do
                    orderBy c.Id

                    including (
                        plasm {
                            for o in tableOf<Order> do
                                where (o.CustomerId = Some c.Id)
                                orderBy o.Id
                                including (paymentsOf o)
                        }
                    )
            }

        let! rows = repo.All customers
        ok rows |> List.map (fun (c, orders) -> c.Name, orders |> List.map (fun (o, payments) -> o.Id, payments.Length)) |> shouldBe __
    })

## About constructors and constraints

Plasm validates no input. What a value may be is the domain's: its constructors (create) refuse bad input, and Plasm reads stored values back through them. What the data may be is the database's: a constraint with PostgreSQL's default name comes back as that field's error, and any other as a database failure.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

/// The field errors a failed transaction reports, or [] for any other outcome.
let fieldErrors (r: Result<'T, TransactionError<unit>>) =
    match r with
    | Error e ->
        match e.Failure with
        | TxFailure.InvalidChanges(_, errors) -> errors
        | _ -> []
    | Ok _ -> []

In [ ]:
koan "A constructor refuses bad input" (fun _ ->
    // Plasm validates no input. Validation is the domain's: an Email
    // (in Shop.fs) can only be made by Email.create, which refuses text
    // without an '@'.
    Email.create "ada.example.com" |> err |> shouldBe __
    (ok (Email.create "ada@example.com")).Value |> shouldBe __)

In [ ]:
koanTask "What the domain refuses is refused when read" (fun repo ->
    task {
        // Plasm reads an Email back through Email.create too. A stored
        // email the domain refuses is a read error naming the field: never
        // an Email that Email.create would not make.
        let broken: Tx<int, unit> = Tx.executeSql $"update customers set email = 'nobody' where id = 3"
        let! _ = repo.Transact broken
        let! customers = repo.All(Query.from (tableOf<Customer>))

        match customers with
        | Error(DbError.Materialization m) -> (m.Property, m.Message) |> shouldBe __
        | other -> failwithf "expected a materialisation error, got %A" other
    })

In [ ]:
koanTask "A duplicate is the field's error" (fun repo ->
    task {
        // The rest is the database's: its constraints. Customer's Email is
        // marked [<Unique>] in Shop.fs, and the table's unique constraint has
        // PostgreSQL's default name, customers_email_key. A duplicate comes
        // back as the Email field's error. fieldErrors is at the top of this lesson.
        // (repo.Transact runs a transaction and Tx.insert is a step that
        // inserts a record: lesson 11 goes deeper.)
        let addAdaAgain: Tx<Customer, unit> = Tx.insert (New.customer "Ada again" "ada@example.com")
        let! r = repo.Transact addAdaAgain
        fieldErrors r |> List.map (fun e -> e.Field, e.Kind) |> shouldBe __
    })

In [ ]:
koanTask "A missing reference is the field's error" (fun repo ->
    task {
        // An order's CustomerId refers to a customer, and the table's foreign
        // key has the default name orders_customer_id_fkey. There is no
        // customer 99.
        let order: Tx<Order, unit> = Tx.insert (New.order (Some(CustomerId 99L)) Draft 1m)
        let! r = repo.Transact order
        fieldErrors r |> List.map (fun e -> e.Field, e.Message) |> shouldBe __
    })

In [ ]:
koanTask "A broken check is the field's error" (fun repo ->
    task {
        // The orders table checks `total >= 0`, in a check constraint with
        // the default name orders_total_check.
        let order: Tx<Order, unit> = Tx.insert (New.order None Draft -1m)
        let! r = repo.Transact order
        fieldErrors r |> List.map (fun e -> e.Kind) |> shouldBe __
    })

In [ ]:
koanTask "Refused changes are never written" (fun repo ->
    task {
        // A refused insert writes nothing: the transaction is rolled back.
        let order: Tx<Order, unit> = Tx.insert (New.order None Draft -1m)
        let! r = repo.Transact order
        Result.isError r |> shouldBe __
        let! n = repo.Count(Query.from (tableOf<Order>))
        ok n |> shouldBe __
    })

In [ ]:
koanTask "A rule across columns is no field's error" (fun repo ->
    task {
        // The orders table has a rule across two columns: a cancelled order
        // needs a reason. Its constraint, orders_cancelled_has_reason, has no
        // field's default name, so breaking it stays a database failure.
        let order: Tx<Order, unit> = Tx.insert (New.order None Cancelled 5m)
        let! r = repo.Transact order

        match (err r).Failure with
        | TxFailure.Failed(DbError.Database f) -> f.Constraint |> shouldBe __
        | other -> failwithf "expected a database failure, got %A" other
    })

## About writes

Every write is a step of a transaction: Tx.insert, Tx.update and Tx.delete take the records themselves, and repo.Transact runs them. An update writes only what differs from the record read, and refuses a change made from a stale copy.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let order (repo: Repo) (id: int64) =
    task {
        let! o = repo.One(plasm { for o in tableOf<Order> do where (o.Id = OrderId id) })
        return (ok o).Value
    }

/// Why a transaction failed, or None if it succeeded.
let failure (r: Result<'T, TransactionError<unit>>) =
    match r with
    | Error e -> Some e.Failure
    | Ok _ -> None

In [ ]:
koanTask "Inserting adds a row" (fun repo ->
    task {
        // Every write goes through a transaction: a description of steps
        // that repo.Transact runs, all or nothing. Tx.insert is a step that
        // inserts a record and gives it back as stored. Tx<Customer, unit>
        // says the transaction gives a Customer and has no reasons of its
        // own to fail (unit); a single step needs to be told that.
        let addDan: Tx<Customer, unit> = Tx.insert (New.customer "Dan" "dan@example.com")
        let! saved = repo.Transact addDan
        (ok saved).Name |> shouldBe __
        let! n = repo.Count(Query.from (tableOf<Customer>))
        ok n |> shouldBe __
    })

In [ ]:
koanTask "An insert assigns the key" (fun repo ->
    task {
        let! saved = repo.Transact(Tx.insert (New.customer "Dan" "dan@example.com"): Tx<Customer, unit>)
        // The database chose Dan's key; the stored record carries it, so it finds the row again.
        let danId = (ok saved).Id
        let! found = repo.One(plasm { for c in tableOf<Customer> do where (c.Id = danId) })
        (ok found |> Option.map (fun c -> c.Name)) |> shouldBe __
    })

In [ ]:
koanTask "An update returns the new record" (fun repo ->
    task {
        // Tx.update (original, changed) changes the record you read into the
        // one you made from it with `with`, and gives it back as stored.
        let! draft = order repo 2L
        let! updated = repo.Transact(Tx.update (draft, { draft with Total = 15m }): Tx<Order, unit>)
        (ok updated).Total |> shouldBe __
        // Records are values: the one you read is not changed.
        draft.Total |> shouldBe __
    })

In [ ]:
koanTask "The version goes up on update" (fun repo ->
    task {
        // tableOf<Order> declares a version field: Version counts the updates of the row.
        let! draft = order repo 2L
        let! updated = repo.Transact(Tx.update (draft, { draft with Status = Published }): Tx<Order, unit>)
        draft.Version |> shouldBe __
        (ok updated).Version |> shouldBe __
    })

In [ ]:
koanTask "Deleting a record" (fun repo ->
    task {
        // Tx.delete deletes the record you read.
        let customers = Query.from (tableOf<Customer>)
        let! cy = repo.One(plasm { for c in tableOf<Customer> do where (c.Name = "Cy") })
        let goodbye: Tx<Customer, unit> = Tx.delete (ok cy).Value
        let! before = repo.Count customers
        ok before |> shouldBe __
        let! _ = repo.Transact goodbye
        let! after = repo.Count customers
        ok after |> shouldBe __
    })

In [ ]:
koanTask "Inserting returns the stored record" (fun repo ->
    task {
        let sprocket = New.product "SPROCKET" "Sprocket"
        let! saved = repo.Transact(Tx.insert sprocket: Tx<Product, unit>)
        // The database generated a key (its value depends on earlier inserts).
        ((ok saved).Id <> ProductId 0L) |> shouldBe __
        sprocket.Id |> shouldBe __
    })

In [ ]:
koanTask "Updates write only the changes" (fun repo ->
    task {
        let! draft = order repo 2L
        let! _ = repo.Transact(Tx.update (draft, { draft with Status = Published }): Tx<Order, unit>)
        let update = sqlSent () |> List.find (fun c -> c.StartsWith "UPDATE")
        // Which columns does the UPDATE set?
        update.Contains "\"status\" = " |> shouldBe __
        update.Contains "\"total\" = " |> shouldBe __
    })

In [ ]:
koanTask "Stale updates are refused" (fun repo ->
    task {
        // An update checks that the row is still the one you read: for a
        // record with a version, by its version. failure is at the top of
        // this lesson.
        let! first = order repo 1L
        let! _ = repo.Transact(Tx.update (first, { first with Total = 31m }): Tx<Order, unit>)
        // `first` still has the old version: this change was made from a stale copy.
        let! again = repo.Transact(Tx.update (first, { first with Total = 32m }): Tx<Order, unit>)
        failure again |> shouldBe __
    })

In [ ]:
koanTask "Without a version, every column is checked" (fun repo ->
    task {
        // A Customer has no version field, so an update checks every column
        // of the record it was made from. Another writer renamed Ada first.
        let! ada = repo.One(plasm { for c in tableOf<Customer> do where (c.Name = "Ada") })
        let ada = (ok ada).Value
        let! _ = repo.Transact(Tx.update (ada, { ada with Name = "Ada L." }): Tx<Customer, unit>)
        let! late = repo.Transact(Tx.update (ada, { ada with Name = "Ada B." }): Tx<Customer, unit>)
        failure late |> shouldBe __
    })

In [ ]:
koanTask "A copy from elsewhere says which version it saw" (fun repo ->
    task {
        // A copy that comes back from a browser or another service carries
        // the version it was read at: expectVersion checks that one instead.
        let! draft = order repo 2L
        let! r = repo.Transact(Tx.update (draft, { draft with Total = 20m }, expectVersion = 7): Tx<Order, unit>)
        failure r |> shouldBe __
        let! after = order repo 2L
        after.Total |> shouldBe __
    })

## About writing an aggregate

An aggregate is written whole, as it is read: inserting an order inserts its lines, an update from an order read writes the difference by key (new lines inserted, changed ones updated, missing ones deleted), and deleting an order deletes its lines. A line's error is the order's.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let order (repo: Repo) (id: OrderId) =
    task {
        let! o = repo.One(plasm { for o in tableOf<Order> do where (o.Id = id) })
        return (ok o).Value
    }

/// How many lines an order has in the database.
let linesOf (repo: Repo) (id: OrderId) =
    task {
        let! n = repo.Count(plasm { for l in tableOf<OrderLine> do where (l.OrderId = id) })
        return ok n
    }

/// A new draft Order for Cy with one line: two Widgets at 10.00 each. Its
/// lines are part of the record, so it is written whole.
let cysOrder =
    { New.order (Some(CustomerId 3L)) Draft 20m with
        Lines = [ New.line (ProductId 1L) 2 10m ] }

In [ ]:
koanTask "Inserting an order inserts its lines" (fun repo ->
    task {
        // An order owns its lines, so one Tx.insert writes the order and
        // then its lines, in one transaction.
        let! _ = repo.Transact(Tx.insert cysOrder: Tx<Order, unit>)
        let! orders = repo.Count(Query.from (tableOf<Order>))
        ok orders |> shouldBe __
        let! lines = repo.Count(Query.from (tableOf<OrderLine>))
        ok lines |> shouldBe __
    })

In [ ]:
koanTask "The stored order holds its stored lines" (fun repo ->
    task {
        let! saved = repo.Transact(Tx.insert cysOrder: Tx<Order, unit>)
        let saved = ok saved
        // The returned Order holds its lines as stored: each was given the
        // new order's key, whatever OrderId it was built with.
        saved.Lines.Length |> shouldBe __
        ((List.head saved.Lines).OrderId = saved.Id) |> shouldBe __
    })

In [ ]:
koanTask "Saved lines are read back" (fun repo ->
    task {
        let! saved = repo.Transact(Tx.insert cysOrder: Tx<Order, unit>)
        let! again = order repo (ok saved).Id
        let line = List.head again.Lines
        line.Quantity.Value |> shouldBe __
        line.UnitPrice |> shouldBe __
    })

In [ ]:
koanTask "Adding a line to an order read" (fun repo ->
    task {
        // Order 2 has one line. Change the list: the line without a key
        // is new, so the update inserts it.
        let! o = order repo (OrderId 2L)
        let added = { o with Lines = o.Lines @ [ New.line (ProductId 1L) 1 10m ] }
        let! _ = repo.Transact(Tx.update (o, added): Tx<Order, unit>)
        let! n = linesOf repo o.Id
        n |> shouldBe __
    })

In [ ]:
koanTask "Lines that did not change are not written" (fun repo ->
    task {
        // An update writes the difference: changing only the total sends
        // no command to order_lines but the one that reads them back.
        let! o = order repo (OrderId 1L)
        let! _ = repo.Transact(Tx.update (o, { o with Total = 40m }): Tx<Order, unit>)
        let lineWrites = sqlSent () |> List.filter (fun c -> c.Contains "order_lines" && not (c.StartsWith "SELECT"))
        lineWrites.Length |> shouldBe __
        let! n = linesOf repo o.Id
        n |> shouldBe __
    })

In [ ]:
koanTask "Lines are matched by key" (fun repo ->
    task {
        // Order 1's lines have keys 1 and 2. A line with a key that is
        // still in the list is updated in place, and keeps its key; a
        // line without one is inserted.
        let! o = order repo (OrderId 1L)
        let first = List.head o.Lines
        let changed = { o with Lines = [ { first with UnitPrice = 9m }; New.line (ProductId 3L) 1 2.5m ] }
        let! saved = repo.Transact(Tx.update (o, changed): Tx<Order, unit>)
        let lines = (ok saved).Lines
        lines.Head.Id |> shouldBe __
        lines.Head.UnitPrice |> shouldBe __
    })

In [ ]:
koanTask "A line left out of the list is deleted" (fun repo ->
    task {
        // The list is the whole truth about an order's lines: a line that
        // was read and is no longer in the list is deleted. No policy is
        // needed, because the list is never a part of the lines.
        let! o = order repo (OrderId 1L)
        let! _ = repo.Transact(Tx.update (o, { o with Lines = [ List.head o.Lines ] }): Tx<Order, unit>)
        let! n = linesOf repo o.Id
        n |> shouldBe __
    })

In [ ]:
koanTask "Changing some lines is changing the list" (fun repo ->
    task {
        // To remove only the Gadget lines (product 2), filter the list in
        // F# and write the order: the lines kept are left as they are.
        let! o = order repo (OrderId 1L)
        let noGadgets = { o with Lines = o.Lines |> List.filter (fun l -> l.ProductId <> ProductId 2L) }
        let! _ = repo.Transact(Tx.update (o, noGadgets): Tx<Order, unit>)
        let! left = repo.All(plasm { for l in tableOf<OrderLine> do where (l.OrderId = OrderId 1L) })
        ok left |> List.map (fun l -> l.ProductId) |> shouldBe __
    })

In [ ]:
koanTask "A line's error is the order's field error" (fun repo ->
    task {
        // The second line's negative price breaks order_lines_unit_price_check.
        // The error names the line by its place in the list, and nothing is
        // written: neither the order nor its first line.
        let lines = [ New.line (ProductId 1L) 1 10m; New.line (ProductId 2L) 1 -1m ]
        let! r = repo.Transact(Tx.insert { New.order None Draft 9m with Lines = lines }: Tx<Order, unit>)

        match (err r).Failure with
        | TxFailure.InvalidChanges(_, errors) -> errors |> List.map (fun e -> e.Field) |> shouldBe __
        | other -> failwithf "expected invalid changes, got %A" other

        let! orders = repo.Count(Query.from (tableOf<Order>))
        ok orders |> shouldBe __
    })

In [ ]:
koanTask "Deleting an order deletes its lines" (fun repo ->
    task {
        // Tx.delete deletes the order read, and the lines it owns.
        let! saved = repo.Transact(Tx.insert cysOrder: Tx<Order, unit>)
        let! _ = repo.Transact(Tx.delete (ok saved): Tx<Order, unit>)
        let! n = linesOf repo (ok saved).Id
        n |> shouldBe __
    })

## About transactions

A transaction is a value made of steps, of type Tx<'T, 'E>: it gives a 'T, or fails with a rejection of its own error type 'E. Running it is explicit; it reads and writes inside one database transaction; the domain decides and the transaction writes what it decided; a failure says where its step is written; retries are opt-in.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let customers (repo: Repo) =
    task {
        let! n = repo.Count(Query.from (tableOf<Customer>))
        return ok n
    }

/// A transaction of one step that inserts Dan. Tx<string, unit>: it gives a
/// string, and has no reasons of its own to fail (unit).
let addDan: Tx<string, unit> =
    transaction {
        let! dan = Tx.insert (New.customer "Dan" "dan@example.com")
        return dan.Name
    }

/// Why the shop refuses a change: its transactions' own error type.
type Refusal =
    | Missing of what: string
    | Refused of reason: string

/// The domain's decision, a plain function: an order may be cancelled with a
/// reason, unless it is already published.
let cancel (reason: string) (order: Order) : Result<Order, string> =
    if order.Status = Published then
        Error "a published order cannot be cancelled"
    else
        Ok { order with Status = Cancelled; CancellationReason = Some reason }

/// Read what the decision needs, let the domain decide, write what it decided.
let cancelOrder (id: OrderId) : Tx<Order, Refusal> =
    transaction {
        let! order = Tx.one (plasm { for o in tableOf<Order> do where (o.Id = id) }) |> Tx.orReject (Missing "order")
        let! cancelled = cancel "duplicate" order |> Tx.ofResult Refused
        return! Tx.update (order, cancelled)
    }

In [ ]:
koanTask "A transaction does nothing until run" (fun repo ->
    task {
        // A transaction is a value describing database steps that succeed or fail together.
        // transaction { } builds it; Tx.insert a step that inserts a record.
        let tx = addDan
        sqlSent().Length |> shouldBe __
        let! n = customers repo
        n |> shouldBe __
    })

In [ ]:
koanTask "Running a transaction commits it" (fun repo ->
    task {
        // repo.Transact runs the steps and commits them: Ok carries what the transaction returns.
        let! r = repo.Transact addDan
        ok r |> shouldBe __
        let! n = customers repo
        n |> shouldBe __
    })

In [ ]:
koanTask "An Error rolls it back" (fun repo ->
    task {
        let tx =
            transaction {
                let! _ = Tx.insert (New.customer "Dan" "dan@example.com")
                // Tx.fail a step that always fails, with a reason.
                return! Tx.fail "changed my mind"
            }

        let! r = repo.Transact tx
        Result.isError r |> shouldBe __
        // A failed step rolls back (undoes) the steps before it.
        let! n = customers repo
        n |> shouldBe __
    })

In [ ]:
koanTask "A step's result flows to the next" (fun repo ->
    task {
        let rename =
            transaction {
                let! dan = Tx.insert (New.customer "Dan" "dan@example.com")
                // let! gives the next step what this one stored: Dan, with his new key.
                let! daniel = Tx.update (dan, { dan with Name = "Daniel" })
                return daniel.Name
            }

        let! r = repo.Transact rename
        ok r |> shouldBe __
    })

In [ ]:
koanTask "Steps compose with let!" (fun repo ->
    task {
        let signUp =
            transaction {
                let! dan = Tx.insert (New.customer "Dan" "dan@example.com")
                let! order = Tx.insert (New.order (Some dan.Id) Draft 1m)
                return order.CustomerId = Some dan.Id
            }

        let! r = repo.Transact signUp
        ok r |> shouldBe __
        let! n = customers repo
        n |> shouldBe __
    })

In [ ]:
koanTask "A failure says where its step is written" (fun repo ->
    task {
        let signUp =
            transaction {
                let! dan = Tx.insert (New.customer "Dan" "dan@example.com")
                // A negative total breaks orders_total_check.
                let! _ = Tx.insert (New.order (Some dan.Id) Draft -1m)
                return ()
            }

        let! r = repo.Transact signUp
        let e = err r
        // A failure gives the step that failed: what it did (Action), and
        // where it is written (File and Line), so steps need no names.
        e.Step.Action |> shouldBe __
        e.RolledBack |> List.map (fun s -> s.Action) |> shouldBe __
        let! n = customers repo
        n |> shouldBe __
    })

In [ ]:
koanTask "Reading inside a transaction" (fun repo ->
    task {
        // Tx.one reads the one row a query matches (or None), and Tx.count
        // counts rows, as steps: inside the transaction, so what they read
        // is what the later steps change.
        let look =
            transaction {
                let! ada = Tx.one (plasm { for c in tableOf<Customer> do where (c.Name = "Ada") })
                let! orders = Tx.count (Query.from (tableOf<Order>))
                return (ada |> Option.map (fun c -> c.Name)), orders
            }

        let! r = repo.Transact(look: Tx<_, unit>)
        ok r |> shouldBe __
    })

In [ ]:
koanTask "A missing row is a rejection" (fun repo ->
    task {
        // A transaction's own reasons to fail are its error type: here
        // Refusal, above under Given. Tx.orReject turns a None into one.
        let zed =
            Tx.one (plasm { for c in tableOf<Customer> do where (c.Name = "Zed") })
            |> Tx.orReject (Missing "customer Zed")

        let! r = repo.Transact zed
        (err r).Failure |> shouldBe __
    })

In [ ]:
koanTask "The domain decides, the transaction writes" (fun repo ->
    task {
        // cancelOrder (above, under Given) reads an order, lets cancel decide,
        // and writes what it decided. Tx.ofResult turns the decision's Error
        // into a rejection. Order 2 is a draft; order 1 is published.
        let! draft = repo.Transact(cancelOrder (OrderId 2L))
        (ok draft).Status |> shouldBe __
        let! published = repo.Transact(cancelOrder (OrderId 1L))
        (err published).Failure |> shouldBe __
    })

In [ ]:
koanTask "An empty transaction is fine" (fun repo ->
    task {
        let! r = repo.Transact(transaction { return 42 })
        r |> shouldBe __
    })

In [ ]:
koan "Retries are opt-in" (fun _ ->
    RetryPolicy.None.MaxAttempts |> shouldBe __
    (RetryPolicy.OnSerializationFailure 3).MaxAttempts |> shouldBe __)

In [ ]:
koanTask "Custom steps use the transaction" (fun repo ->
    task {
        let audit =
            transaction {
                let! _ =
                    Tx.run (fun ctx ->
                        task {
                            use cmd = ctx.CreateCommand()
                            cmd.CommandText <- "insert into products(sku, name) values ('AUDIT', 'Audit')"
                            let! _ = cmd.ExecuteNonQueryAsync(ctx.CancellationToken)
                            return (Ok(): Result<unit, string>)
                        })

                return! Tx.fail "changed my mind"
            }

        let! r = repo.Transact audit
        (err r).Step.Action |> shouldBe __
        let! audited = repo.Count(plasm { for p in tableOf<Product> do where (p.Sku = "AUDIT") })
        ok audited |> shouldBe __
    })

In [ ]:
koanTask "Effects outside the database are not rolled back" (fun repo ->
    task {
        let emailsSent = ref 0

        let tx =
            transaction {
                let! _ =
                    Tx.run (fun _ ->
                        task {
                            emailsSent.Value <- emailsSent.Value + 1
                            return (Ok(): Result<unit, string>)
                        })

                return! Tx.fail "changed my mind"
            }

        let! _ = repo.Transact tx
        emailsSent.Value |> shouldBe __
    })

## About joins

Joins return combined rows from one command; left joins give the missing side as None.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koanTask "The simplest inner join" (fun repo ->
    task {
        // join pairs each row with the rows of another table whose key matches (an inner join).
        // Every OrderLine has a ProductId, so every line finds its Product.
        let linesWithProducts =
            plasm {
                for l in tableOf<OrderLine> do
                    join p in tableOf<Product> on (l.ProductId = p.Id)
                    orderBy l.Id
            }

        let! rows = repo.All linesWithProducts
        ok rows |> List.map (fun (_, p) -> p.Name) |> shouldBe __
    })

In [ ]:
koanTask "Selecting a pair's fields" (fun repo ->
    task {
        // select can name fields from either side of the join.
        let orderOne =
            plasm {
                for l in tableOf<OrderLine> do
                    join p in tableOf<Product> on (l.ProductId = p.Id)
                    where (l.OrderId = OrderId 1L)
                    orderBy l.Id
                    select (p.Name, l.UnitPrice)
            }

        let! rows = repo.All orderOne
        ok rows |> shouldBe __
    })

In [ ]:
koanTask "Counting joined rows" (fun repo ->
    task {
        // A join gives one row for each matching pair, so an Order appears once per Payment.
        let paid =
            plasm {
                for o in tableOf<Order> do
                    join p in tableOf<Payment> on (o.Id = p.OrderId)
                    orderBy p.Id
            }

        let! n = repo.Count paid
        ok n |> shouldBe __
    })

In [ ]:
koanTask "A left join's missing side is None" (fun repo ->
    task {
        // leftJoin keeps rows with no match; the other side is an option, None when missing.
        let fourth =
            plasm {
                for o in tableOf<Order> do
                    leftJoin c in tableOf<Customer> on (o.CustomerId = Some c.Id)
                    where (o.Id = OrderId 4L)
                    select (o.Id, c)
            }

        let! row = repo.One fourth
        let (_, customer) = (ok row).Value
        customer |> shouldBe __
    })

In [ ]:
koanTask "An inner join gives pairs" (fun repo ->
    task {
        let pairs =
            plasm {
                for o in tableOf<Order> do
                    join c in tableOf<Customer> on (o.CustomerId = Some c.Id)
                    orderBy o.Id
                    select (o.Id, c)
            }

        // A whole Customer can be selected; a whole Order cannot, because
        // it is read with its lines, which a join does not read. Select
        // the order's fields instead.
        let! rows = repo.All pairs
        ok rows |> List.map (fun (id, c) -> id, c.Name) |> shouldBe __
    })

In [ ]:
koanTask "A left join keeps every row" (fun repo ->
    task {
        let all =
            plasm {
                for o in tableOf<Order> do
                    leftJoin c in tableOf<Customer> on (o.CustomerId = Some c.Id)
                    orderBy o.Id
                    select (o.Id, c)
            }

        let! rows = repo.All all
        ok rows |> List.map (fun (id, c) -> id, c |> Option.map (fun c -> c.Name)) |> List.last |> shouldBe __
    })

In [ ]:
koanTask "Join keys are typed" (fun repo ->
    task {
        // The keys must have the same type: an optional CustomerId on the Order side.
        let pairs =
            plasm {
                for o in tableOf<Order> do
                    join c in tableOf<Customer> on (o.CustomerId = __)
                    orderBy o.Id
            }

        let! rows = repo.Count pairs
        ok rows |> shouldBe 3L
    })

In [ ]:
koanTask "Filters on joined columns run in SQL" (fun repo ->
    task {
        let adas =
            plasm {
                for o in tableOf<Order> do
                    join c in tableOf<Customer> on (o.CustomerId = Some c.Id)
                    where (c.Name = "Ada")
                    orderBy o.Id
                    select o.Id
            }

        (ok (repo.Explain adas)).Sql.Contains "WHERE" |> shouldBe __
        let! ids = repo.All adas
        ok ids |> shouldBe __
    })

In [ ]:
koanTask "Joined queries compose" (fun repo ->
    task {
        let withCustomers = plasm { for o in tableOf<Order> do join c in tableOf<Customer> on (o.CustomerId = Some c.Id); orderBy o.Id }
        let large (q: Query<Order * Customer>) = plasm { for (o, c) in q do where (o.Total > 20m); select c.Name }
        let! rows = repo.All(large withCustomers)
        ok rows |> shouldBe __
    })

## About aggregates and grouping

Aggregates are typed values computed in the database; grouping summarises without transferring rows.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koanTask "Counting rows with Agg.count" (fun repo ->
    task {
        // An aggregate summarises many rows as one value, computed by the database.
        // Agg.count () is the number of rows; One reads the single row it gives.
        let! n = repo.One(plasm { for o in tableOf<Order> do where (o.Status = Published); select (Agg.count ()) })
        (ok n).Value |> shouldBe __
    })

In [ ]:
koanTask "Adding up with Agg.sum" (fun repo ->
    task {
        // Agg.sum adds up a column over the rows the query matches.
        let! total = repo.One(plasm { for o in tableOf<Order> do where (o.Status = Published); select (Agg.sum o.Total) })
        (ok total).Value |> shouldBe __
    })

In [ ]:
koanTask "The largest with Agg.max" (fun repo ->
    task {
        // Agg.max is the largest value: an option, because there might be no rows.
        let! largest = repo.One(plasm { for o in tableOf<Order> do select (Agg.max o.Total) })
        (ok largest).Value |> shouldBe __
    })

In [ ]:
koanTask "Grouping by one key" (fun repo ->
    task {
        // groupBy puts rows with the same key in one group; select gives one row per group.
        let linesPerOrder =
            plasm {
                for l in tableOf<OrderLine> do
                    groupBy l.OrderId
                    orderBy l.OrderId
                    select (l.OrderId, Agg.count ())
            }

        let! rows = repo.All linesPerOrder
        ok rows |> shouldBe __
    })

In [ ]:
koanTask "Counting a page counts the page" (fun repo ->
    task {
        let! n = repo.Count(plasm { for o in tableOf<Order> do orderBy o.Id; take 2 })
        ok n |> shouldBe __
    })

In [ ]:
koanTask "Sum and average are typed" (fun repo ->
    task {
        let! totals = repo.One(plasm { for o in tableOf<Order> do select (Agg.sum o.Total, Agg.avg o.Total) })
        // The four totals are 30.00, 12.50, 99.00 and 5.00.
        (ok totals).Value |> shouldBe __
    })

In [ ]:
koanTask "No rows is explicit" (fun repo ->
    task {
        let! r = repo.One(plasm { for o in tableOf<Order> do where (o.Total > 1000m); select (Agg.sum o.Total, Agg.max o.Total) })
        (ok r).Value |> shouldBe __
    })

In [ ]:
koanTask "Grouping by a key" (fun repo ->
    task {
        let perCustomer =
            plasm {
                for o in tableOf<Order> do
                    groupBy o.CustomerId
                    orderBy o.CustomerId
                    select (o.CustomerId, Agg.count ())
            }

        let! rows = repo.All perCustomer
        ok rows |> shouldBe __
    })

In [ ]:
koanTask "Filtering groups" (fun repo ->
    task {
        let repeat =
            plasm {
                for o in tableOf<Order> do
                    groupBy o.CustomerId
                    having (Agg.count () > 1L)
                    select (o.CustomerId, Agg.count ())
            }

        let! rows = repo.All repeat
        ok rows |> shouldBe __
    })

In [ ]:
koanTask "Each value once" (fun repo ->
    task {
        // selectDistinct keeps each selected value once, as List.distinct
        // would: the order with no customer gives None, one value of its
        // own. Agg.countDistinct counts the distinct values that are not
        // None, as SQL's COUNT(DISTINCT ...) does.
        let! customers = repo.All(plasm { for o in tableOf<Order> do selectDistinct o.CustomerId })
        (ok customers).Length |> shouldBe __

        let! counted = repo.One(plasm { for o in tableOf<Order> do select (Agg.countDistinct o.CustomerId) })
        ok counted |> shouldBe (Some __)
    })

In [ ]:
koanTask "Grouping mistakes are refused" (fun repo ->
    task {
        // Status is neither grouped nor aggregated.
        let wrong = plasm { for o in tableOf<Order> do groupBy o.CustomerId; select (o.Status, Agg.count ()) }
        let! r = repo.All wrong
        Result.isError r |> shouldBe __
        sqlSent().Length |> shouldBe __
    })

## About links and owned records

An article links to shared tags through a join table: its Tags are read with it, and writing the article writes only its links. An account owns at most one profile, its own row, read and written with it as an option.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let names (tags: Tag list) = tags |> List.map (fun t -> t.Name)

let article (repo: Repo) (id: int64) =
    task {
        let! a = repo.One(plasm { for a in tableOf<Article> do where (a.Id = ArticleId id) })
        return (ok a).Value
    }

let account (repo: Repo) (name: string) =
    task {
        let! a = repo.One(plasm { for a in tableOf<Account> do where (a.Name = name) })
        return (ok a).Value
    }

let tag id name = { Id = TagId id; Name = name }

/// The field errors a failed transaction reports, or [] for any other outcome.
let fieldErrors (r: Result<'T, TransactionError<unit>>) =
    match r with
    | Error e ->
        match e.Failure with
        | TxFailure.InvalidChanges(_, errors) -> errors
        | _ -> []
    | Ok _ -> []

In [ ]:
koanTask "An article is read with its tags" (fun repo ->
    task {
        // A Tag is shared: many articles can link to the same one. An article
        // does not own its tags, only its links to them: rows of the
        // article_tags table (tableOf<Article> says links ... "article_tags").
        // The tags are read with the article, in the order of their key.
        // In the starting data, article 2 ('Deep dive') links to tags 1 and 3.
        let! deepDive = article repo 2L
        deepDive.Tags |> names |> shouldBe __
    })

In [ ]:
koanTask "No tags is an empty list" (fun repo ->
    task {
        // Article 3 links to no tag.
        let! empty = article repo 3L
        empty.Tags |> shouldBe __
    })

In [ ]:
koanTask "Every article's tags come in one more command" (fun repo ->
    task {
        // One command reads the articles, and one more reads the tags of
        // all of them, through the join table.
        let! articles = repo.All(plasm { for a in tableOf<Article> do orderBy a.Id })
        (ok articles).Head.Tags |> names |> shouldBe __
        sqlSent().Length |> shouldBe __
    })

In [ ]:
koanTask "Filtering articles by their tags" (fun repo ->
    task {
        // List.exists over a linked list is a filter the database runs,
        // keeping each matching article once.
        let fsharp =
            plasm {
                for a in tableOf<Article> do
                    where (a.Tags |> List.exists (fun t -> t.Name = "fsharp"))
                    orderBy a.Id
            }

        let! rows = repo.All fsharp
        ok rows |> List.map (fun a -> a.Title) |> shouldBe __
    })

In [ ]:
koanTask "Changing the tags writes only links" (fun repo ->
    task {
        // Article 1 links to fsharp and sql. Give it the ops tag instead:
        // the update rewrites its links, and writes no Tag.
        let! intro = article repo 1L
        let! updated = repo.Transact(Tx.update (intro, { intro with Tags = [ tag 4L "ops" ] }): Tx<Article, unit>)
        (ok updated).Tags |> names |> shouldBe __
        let! tags = repo.Count(Query.from (tableOf<Tag>))
        ok tags |> shouldBe __
    })

In [ ]:
koanTask "Adding and removing one tag" (fun repo ->
    task {
        // The list is the article's whole set of links: change it in F#.
        let! intro = article repo 1L
        let retagged = (intro.Tags |> List.filter (fun t -> t.Name <> "sql")) @ [ tag 3L "draft" ]
        let! _ = repo.Transact(Tx.update (intro, { intro with Tags = retagged }): Tx<Article, unit>)
        let! reloaded = article repo 1L
        reloaded.Tags |> names |> shouldBe __
    })

In [ ]:
koanTask "A link to a missing tag is the field's error" (fun repo ->
    task {
        // There is no tag 99. The join table's foreign key refuses the link,
        // and it comes back as the Tags field's error. fieldErrors is at the
        // top of this lesson.
        let! intro = article repo 1L
        let! r = repo.Transact(Tx.update (intro, { intro with Tags = [ tag 99L "nope" ] }): Tx<Article, unit>)
        fieldErrors r |> List.map (fun e -> e.Field, e.Kind) |> shouldBe __
    })

In [ ]:
koanTask "A tag is changed as a tag" (fun repo ->
    task {
        // A tag belongs to its own aggregate, so renaming it through an
        // article is refused: write the Tag itself.
        let! intro = article repo 1L
        let renamed = intro.Tags |> List.map (fun t -> { t with Name = t.Name.ToUpperInvariant() })
        let! r = repo.Transact(Tx.update (intro, { intro with Tags = renamed }): Tx<Article, unit>)
        fieldErrors r |> List.map (fun e -> e.Field) |> shouldBe __
    })

In [ ]:
koanTask "Deleting an article deletes its links, not its tags" (fun repo ->
    task {
        // article_tags has no record of its own; hand-written SQL counts it.
        let! intro = article repo 1L
        let! _ = repo.Transact(Tx.delete intro: Tx<Article, unit>)
        let! links = repo.Sql<int64>($"select count(*) from article_tags")
        ok links |> shouldBe __
        let! tags = repo.Count(Query.from (tableOf<Tag>))
        ok tags |> shouldBe __
    })

In [ ]:
koanTask "An account's profile is an option" (fun repo ->
    task {
        // An Account owns at most one Profile: its own row in profiles. A
        // Profile holds its account's key (AccountId), so the account owns it.
        // It is read with the account, as an option.
        let! ada = account repo "ada"
        ada.Profile |> Option.map (fun p -> p.Bio) |> shouldBe __
    })

In [ ]:
koanTask "An account without a profile" (fun repo ->
    task {
        // In the starting data, only account 1 ('ada') has a profile.
        let! bob = account repo "bob"
        bob.Profile |> shouldBe __
    })

In [ ]:
koanTask "Adding a profile inserts it" (fun repo ->
    task {
        // Changing None to Some inserts the profile, for this account.
        let! bob = account repo "bob"
        let profile = { Id = ProfileId 0L; AccountId = bob.Id; Bio = "Builds things" }
        let! _ = repo.Transact(Tx.update (bob, { bob with Profile = Some profile }): Tx<Account, unit>)
        let! profiles = repo.Count(Query.from (tableOf<Profile>))
        ok profiles |> shouldBe __
    })

In [ ]:
koanTask "Removing the profile deletes it" (fun repo ->
    task {
        // Changing Some to None deletes the profile row.
        let! ada = account repo "ada"
        let! _ = repo.Transact(Tx.update (ada, { ada with Profile = None }): Tx<Account, unit>)
        let! profiles = repo.Count(Query.from (tableOf<Profile>))
        ok profiles |> shouldBe __
    })

In [ ]:
koanTask "Filtering through an owned record" (fun repo ->
    task {
        // To keep the accounts whose profile says something, ask whether a
        // profile of this account exists that says it.
        let writers =
            plasm {
                for a in tableOf<Account> do
                    where (exists (plasm { for p in tableOf<Profile> do where (p.AccountId = a.Id && p.Bio.Contains "compiler") }))
            }

        let! rows = repo.All writers
        ok rows |> List.map (fun a -> a.Name) |> shouldBe __
    })

## About bulk writes

Bulk updates and deletes are driven by typed queries; inserts take many records in one command.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koanTask "A bulk update returns a count" (fun repo ->
    task {
        // A bulk update changes every row a query matches, in one SQL command.
        // The function says what each matching row becomes. Like every write,
        // it is a step of a transaction: Tx.execute, run by repo.Transact.
        let gizmos = plasm { for p in tableOf<Product> do where (p.Sku = "GIZMO") }
        let! changed = repo.Transact(Tx.execute (Bulk.update (gizmos, fun p -> { p with Name = "Gizmo Mark II" })): Tx<_, unit>)
        // The result is how many rows were changed.
        ok changed |> shouldBe __
    })

In [ ]:
koanTask "A bulk update can match nothing" (fun repo ->
    task {
        let sprockets = plasm { for p in tableOf<Product> do where (p.Sku = "SPROCKET") }
        let! changed = repo.Transact(Tx.execute (Bulk.update (sprockets, fun p -> { p with Name = "Sprocket" })): Tx<_, unit>)
        ok changed |> shouldBe __
    })

In [ ]:
koanTask "A bulk delete returns a count" (fun repo ->
    task {
        // A bulk delete removes every row a query matches, and returns how many.
        let ofOrder1 = plasm { for p in tableOf<Payment> do where (p.OrderId = OrderId 1L) }
        let! removed = repo.Transact(Tx.execute (Bulk.delete ofOrder1): Tx<_, unit>)
        ok removed |> shouldBe __
    })

In [ ]:
koanTask "Deleting every payment" (fun repo ->
    task {
        // Query.from with no filter matches every row of the table.
        let! removed = repo.Transact(Tx.execute (Bulk.delete (Query.from (tableOf<Payment>))): Tx<_, unit>)
        ok removed |> shouldBe __
    })

In [ ]:
koanTask "Updating what a query matches" (fun repo ->
    task {
        let drafts = plasm { for o in tableOf<Order> do where (o.Status = Draft) }
        let! changed = repo.Transact(Tx.execute (Bulk.update (drafts, fun o -> { o with Status = Published })): Tx<_, unit>)
        ok changed |> shouldBe __
        let! published = repo.Count(plasm { for o in tableOf<Order> do where (o.Status = Published) })
        ok published |> shouldBe __
    })

In [ ]:
koanTask "Updating from the current row" (fun repo ->
    task {
        let! _ = repo.Transact(Tx.execute (Bulk.update (Query.from (tableOf<Order>), fun o -> { o with Total = o.Total * 2m })): Tx<_, unit>)
        let! total = repo.One(plasm { for o in tableOf<Order> do select (Agg.sum o.Total) })
        (ok total).Value |> shouldBe __
    })

In [ ]:
koanTask "Deleting what a query matches" (fun repo ->
    task {
        let failed = plasm { for p in tableOf<Payment> do where (p.Status = Failed) }
        let! removed = repo.Transact(Tx.execute (Bulk.delete failed): Tx<_, unit>)
        ok removed |> shouldBe __
        let! left = repo.Count(Query.from (tableOf<Payment>))
        ok left |> shouldBe __
    })

In [ ]:
koanTask "Inserting many records" (fun repo ->
    task {
        let parts = [ New.product "BOLT" "Bolt"; New.product "NUT" "Nut"; New.product "WASHER" "Washer" ]
        let! inserted = repo.Transact(Tx.execute (Bulk.insert parts): Tx<_, unit>)
        ok inserted |> List.map (fun p -> p.Sku) |> shouldBe __
        sqlSent().Length |> shouldBe __
    })

In [ ]:
koanTask "Inserting only what is missing" (fun repo ->
    task {
        let catalogue = [ New.product "WIDGET" "Widget"; New.product "SPROCKET" "Sprocket" ]
        let! inserted = repo.Transact(Tx.execute (Bulk.insertMissing (catalogue, fun p -> p.Sku)): Tx<_, unit>)
        ok inserted |> List.map (fun p -> p.Sku) |> shouldBe __
    })

In [ ]:
koanTask "Bulk writes are steps like any other" (fun repo ->
    task {
        // A bulk write is one step of a transaction: when a later step
        // fails, it is rolled back with the rest.
        let tx =
            transaction {
                let! _ = Tx.execute (Bulk.delete (Query.from (tableOf<Payment>)))
                return! Tx.fail "changed my mind"
            }

        let! r = repo.Transact tx
        Result.isError r |> shouldBe __
        let! left = repo.Count(Query.from (tableOf<Payment>))
        ok left |> shouldBe __
    })

In [ ]:
koanTask "Bulk writes can be inspected" (fun repo ->
    task {
        let plan = repo.Explain(Bulk.delete (plasm { for p in tableOf<Payment> do where (p.Status = Failed) }))
        (ok plan).Sql.StartsWith "DELETE FROM \"payments\"" |> shouldBe __
        sqlSent().Length |> shouldBe __
    })

## About hand-written SQL

When a query is easier in SQL, write SQL; interpolated values are always parameters and rows still come back typed.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

In [ ]:
koanTask "A hand-written SELECT returns records" (fun repo ->
    task {
        // repo.Sql<'R> runs SQL you write ($"..." is an interpolated string) and reads
        // each row into an 'R. Columns are matched to fields by name.
        let! products = repo.Sql<Product>($"SELECT * FROM products ORDER BY id")
        ok products |> List.map (fun p -> p.Sku) |> shouldBe __
    })

In [ ]:
koanTask "No matching rows is an empty list" (fun repo ->
    task {
        let! products = repo.Sql<Product>($"SELECT * FROM products WHERE sku = 'SPROCKET'")
        ok products |> shouldBe __
    })

In [ ]:
koanTask "SQL can return a single number" (fun repo ->
    task {
        // A row with one column can be read as that column's value.
        let! counts = repo.Sql<int64>($"SELECT count(*) FROM products")
        ok counts |> shouldBe __
    })

In [ ]:
koanTask "SQL can return pairs" (fun repo ->
    task {
        // A row with two columns can be read as a tuple: the first column, then the second.
        let! pairs = repo.Sql<string * string>($"SELECT sku, name FROM products ORDER BY id")
        ok pairs |> List.head |> shouldBe __
    })

In [ ]:
koanTask "An interpolated value becomes a parameter" (fun repo ->
    task {
        // A parameter is a value sent beside the SQL text, never pasted into it.
        let sku = "GADGET"
        let! found = repo.Sql<Product>($"SELECT * FROM products WHERE sku = {sku}")
        ok found |> List.map (fun p -> p.Name) |> shouldBe __
        (List.last (sqlSent ())).Contains "GADGET" |> shouldBe __
    })

In [ ]:
koanTask "SQL can return mapped records" (fun repo ->
    task {
        // Rows are read into a record of the model by column name. (Not a
        // whole Order: an Order is read with its lines, which a query of
        // the orders table alone does not read.)
        let minimum = 50m
        let! big = repo.Sql<Payment>($"SELECT * FROM payments WHERE amount >= {minimum} ORDER BY id")
        ok big |> List.map (fun p -> p.Id) |> shouldBe __
    })

In [ ]:
koanTask "SQL can return tuples and scalars" (fun repo ->
    task {
        let! names = repo.Sql<string>($"SELECT name FROM customers ORDER BY name")
        ok names |> shouldBe __
        let! perStatus = repo.Sql<string * int64>($"SELECT status, count(*) FROM orders GROUP BY status ORDER BY status")
        ok perStatus |> List.head |> shouldBe __
    })

In [ ]:
koanTask "Interpolation is parameters" (fun repo ->
    task {
        let name = "Ada'; drop table customers; --"
        let! found = repo.Sql<Customer>($"SELECT * FROM customers WHERE name = {name}")
        ok found |> shouldBe __
        (List.last (sqlSent ())).Contains "@p0" |> shouldBe __
    })

In [ ]:
koanTask "A fragment inside a typed query" (fun repo ->
    task {
        let startingWithA = plasm { for c in tableOf<Customer> do where (Raw.fragment<bool> "{0} ILIKE {1}" (c.Name, "a%")) }
        let! rows = repo.All startingWithA
        ok rows |> List.map (fun c -> c.Name) |> shouldBe __
    })

In [ ]:
koanTask "Commands return affected rows" (fun repo ->
    task {
        // A hand-written command is a write, so it is a transaction step:
        // Tx.executeSql gives how many rows it changed.
        let archive: Tx<int, unit> =
            Tx.executeSql $"update orders set status = 'cancelled', cancellation_reason = 'archived' where status = 'published'"

        let! n = repo.Transact archive
        ok n |> shouldBe __
    })

In [ ]:
koanTask "Hand-written SQL inside a transaction" (fun repo ->
    task {
        // Tx.sql reads with hand-written SQL as a step, so it sees what the
        // steps before it wrote, before anything is committed.
        let tx =
            transaction {
                let! _ = Tx.executeSql $"delete from payments where status = 'failed'"
                let! left = Tx.sql<int64, unit> $"select count(*) from payments"
                return List.head left
            }

        let! r = repo.Transact tx
        ok r |> shouldBe __
    })

## About prepared queries

A hot query can be built and planned once and bound to new arguments on each call.

In [ ]:
open System
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let ordersOf =
    Query.prepare (fun (customer: Arg<CustomerId>) ->
        plasm {
            for o in tableOf<Order> do
                where (o.CustomerId = Some customer.Value)
                orderBy o.Id
        })

/// A prepared query: built once, then called with a SKU to give the query for that SKU.
/// An Arg<string> stands for the value given at each call; `.Value` reads it inside the query.
let productsWithSku =
    Query.prepare (fun (sku: Arg<string>) -> plasm { for p in tableOf<Product> do where (p.Sku = sku.Value) })

let names (r: Result<Product list, DbError>) = ok r |> List.map (fun p -> p.Name)

In [ ]:
koanTask "A prepared query takes an argument" (fun repo ->
    task {
        // productsWithSku (top of this file) is a function: give it a SKU, get a query.
        let! gizmos = repo.All(productsWithSku "GIZMO")
        names gizmos |> shouldBe __
    })

In [ ]:
koanTask "One prepared query, two values" (fun repo ->
    task {
        let! widgets = repo.All(productsWithSku "WIDGET")
        let! gadgets = repo.All(productsWithSku "GADGET")
        names widgets |> shouldBe __
        names gadgets |> shouldBe __
    })

In [ ]:
koanTask "A prepared query can be counted" (fun repo ->
    task {
        // ordersOf (top of this file) is prepared with a CustomerId. Customer 3 is Cy.
        let! cys = repo.Count(ordersOf (CustomerId 3L))
        ok cys |> shouldBe __
    })

In [ ]:
koanTask "Built once, run many times" (fun repo ->
    task {
        let! adas = repo.Count(ordersOf (CustomerId 1L))
        let! bobs = repo.Count(ordersOf (CustomerId 2L))
        (ok adas, ok bobs) |> shouldBe __
        // Did the two counts send the same SQL text?
        ((sqlSent ()).[0] = (sqlSent ()).[1]) |> shouldBe __
    })

In [ ]:
koan "Arguments are parameters" (fun repo ->
    let sql = (ok (repo.Explain(ordersOf (CustomerId 42L)))).Sql
    sql.Contains "42" |> shouldBe __)

In [ ]:
koanTask "Paging by arguments" (fun repo ->
    task {
        let page =
            Query.prepare2 (fun (size: Arg<int>) (offset: Arg<int>) ->
                plasm {
                    for o in tableOf<Order> do
                        orderBy o.Id
                        skip offset.Value
                        take size.Value
                })

        let! second = repo.All(page 2 2)
        ok second |> List.map (fun o -> o.Id) |> shouldBe __
    })

In [ ]:
koanTask "Prepared queries still compose" (fun repo ->
    task {
        let publishedOf c = plasm { for o in ordersOf c do where (o.Status = Published) }
        let! n = repo.Count(publishedOf (CustomerId 1L))
        ok n |> shouldBe __
    })

In [ ]:
koan "Arguments are read only inside the query" (fun _ ->
    let refused =
        try
            Query.prepare (fun (customer: Arg<CustomerId>) ->
                let id = customer.Value // read while the query is being built
                plasm { for o in tableOf<Order> do where (o.CustomerId = Some id) })
            |> ignore

            false
        with :? InvalidOperationException ->
            true

    refused |> shouldBe __)

In [ ]:
koanTask "An exists takes an argument too" (fun repo ->
    task {
        // An inner query inside exists can read the prepared query's
        // arguments: the orders with a payment of at least an amount.
        let paidAtLeast =
            Query.prepare (fun (least: Arg<decimal>) ->
                plasm {
                    for o in tableOf<Order> do
                        where (exists (plasm { for p in tableOf<Payment> do where (p.OrderId = o.Id && p.Amount >= least.Value) }))
                        orderBy o.Id
                })

        let! big = repo.All(paidAtLeast 50m)
        ok big |> List.map (fun o -> o.Id) |> shouldBe __
        let! any = repo.All(paidAtLeast 0m)
        ok any |> List.map (fun o -> o.Id) |> shouldBe __
    })

## About failures

Expected failures are values you can match on; Plasm's error cases are always qualified, so your domain keeps its own names.

In [ ]:
open System.Threading.Tasks
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

/// Your own union may use the names Plasm uses for its errors.
type Problem =
    | Database
    | Translation

/// A domain error a transaction step can fail with.
type StockError = OutOfStock of sku: string

In [ ]:
koanTask "Nothing found is not a failure" (fun repo ->
    task {
        // Every repository call returns a Result: Ok with the value, or Error with what went wrong.
        let! r = repo.One(plasm { for c in tableOf<Customer> do where (c.Name = "Zed") })
        r |> shouldBe __
    })

In [ ]:
koanTask "A failure is an Error you can match on" (fun repo ->
    task {
        // customers.email is unique ([<Unique>] in Shop.fs), and Ada already has this email.
        let! r = repo.Transact(Tx.insert (New.customer "Ada again" "ada@example.com"): Tx<Customer, unit>)

        let outcome =
            match r with
            | Ok _ -> "saved"
            | Error _ -> "refused"

        outcome |> shouldBe __
    })

In [ ]:
koanTask "A failed write says what kind of failure" (fun repo ->
    task {
        // TxFailure.Failed(DbError.Database f) is a failure reported by the database.
        // Its Kind is a DbFailureKind: UniqueViolation, ForeignKeyViolation,
        // CheckViolation, NotNullViolation, and others. The orders table's rule
        // orders_cancelled_has_reason (a cancelled order needs a reason) spans two
        // columns, so it is no single field's error.
        let! r = repo.Transact(Tx.insert (New.order None Cancelled 5m): Tx<Order, unit>)

        match (err r).Failure with
        | TxFailure.Failed(DbError.Database f) -> f.Kind |> shouldBe __
        | other -> failwithf "expected a database failure, got %A" other
    })

In [ ]:
koanTask "Your refusals and Plasm's are apart" (fun repo ->
    task {
        // A transaction fails with its own reasons (TxFailure.Rejected, of its
        // own error type) or with Plasm's: the database's (TxFailure.Failed),
        // a stale copy (TxFailure.Conflict) or a refused field
        // (TxFailure.InvalidChanges). whose tells them apart.
        let whose (r: Result<Order, TransactionError<StockError>>) =
            match r with
            | Ok _ -> "nobody's"
            | Error e ->
                match e.Failure with
                | TxFailure.Rejected _ -> "the shop's"
                | _ -> "Plasm's"

        let! outOfStock = repo.Transact(Tx.fail (OutOfStock "GIZMO"))
        let! cancelled = repo.Transact(Tx.insert (New.order None Cancelled 5m))
        whose outOfStock |> shouldBe __
        whose cancelled |> shouldBe __
    })

In [ ]:
koanTask "Failures are values" (fun repo ->
    task {
        let kept = ref Unchecked.defaultof<Repo>

        do!
            repo.Sandbox(fun inner ->
                kept.Value <- inner
                Task.FromResult())

        // The sandbox has ended; its repository returns an error rather than throwing.
        let! r = kept.Value.All(Query.from (tableOf<Customer>))

        match r with
        | Error(DbError.Database f) -> f.Kind |> shouldBe __
        | other -> failwithf "expected a database error, got %A" other
    })

In [ ]:
koan "Error cases are qualified" (fun _ ->
    let classify (e: DbError) =
        match e with
        | DbError.Database _ -> Database
        | _ -> Translation

    classify (DbError.Translation(TranslationError.InvalidQuery "example")) |> shouldBe __)

In [ ]:
koanTask "Unreadable rows are named" (fun repo ->
    task {
        let lose: Tx<int, unit> = Tx.executeSql $"update orders set status = 'lost' where id = 1"
        let! _ = repo.Transact lose
        let! r = repo.All(Query.from (tableOf<Order>))

        match r with
        | Error(DbError.Materialization m) -> (m.Model, m.Property) |> shouldBe __
        | other -> failwithf "expected a materialisation error, got %A" other
    })

In [ ]:
koanTask "SQL mismatches name the column" (fun repo ->
    task {
        // Customer needs an email column, and this query does not return one.
        let! r = repo.Sql<Customer>($"SELECT id, name FROM customers")

        match r with
        | Error e -> (string e).Contains "Email" |> shouldBe __
        | Ok rows -> failwithf "expected an error, got %A" rows
    })

In [ ]:
koanTask "Too many rows is about the data" (fun repo ->
    task {
        let! r = repo.One(plasm { for o in tableOf<Order> do where (o.Status = Published) })
        r |> shouldBe __
    })

In [ ]:
koanTask "Rejections keep their type" (fun repo ->
    task {
        let! r = repo.Transact(transaction { return! Tx.fail (OutOfStock "GIZMO") })

        // A transaction's own reasons to fail have its own type, here StockError.
        match (err r).Failure with
        | TxFailure.Rejected reason -> reason |> shouldBe __
        | other -> failwithf "expected a rejection, got %A" other
    })

In [ ]:
koan "Every union is qualified" (fun _ ->
    // PaymentStatus (in Shop.fs) has a case Failed, and so does Plasm's
    // TxFailure. Plasm's cases are always written with their type name, so
    // a plain Failed is your domain's, and TxFailure.Failed is Plasm's.
    let paid (s: PaymentStatus) =
        match s with
        | Succeeded -> true
        | Pending
        | Failed -> false

    let databaseFailed (f: TxFailure<unit>) =
        match f with
        | TxFailure.Failed _ -> true
        | _ -> false

    paid Failed |> shouldBe __
    databaseFailed (TxFailure.Conflict "Order") |> shouldBe __)

## About testing with Plasm

Test against the real database inside sandboxes that always roll back. Check the model against the schema and every query value against the model at start-up or in a test, before anything runs, and print the tables the model expects.

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let customers (repo: Repo) =
    task {
        let! n = repo.Count(Query.from (tableOf<Customer>))
        return ok n
    }

/// A step that inserts Dan, a new customer.
let addDan: Tx<Customer, unit> = Tx.insert (New.customer "Dan" "dan@example.com")

In [ ]:
koanTask "A sandbox gives back its body's result" (fun repo ->
    task {
        // A sandbox runs a function with its own repository and undoes all its changes when the
        // function ends. The sandbox returns whatever the function returns.
        // `customers` (top of this file) counts the Customers.
        let! counted = repo.Sandbox(fun repo -> customers repo)
        counted |> shouldBe __
    })

In [ ]:
koanTask "An insert is visible inside its sandbox" (fun repo ->
    task {
        let! inside =
            repo.Sandbox(fun repo ->
                task {
                    let! _ = repo.Transact addDan
                    return! customers repo
                })

        inside |> shouldBe __
    })

In [ ]:
koanTask "A sandbox's insert is gone afterwards" (fun repo ->
    task {
        let! _ = repo.Sandbox(fun repo -> repo.Transact addDan)
        // The sandbox has ended: count with the outer repository.
        let! after = customers repo
        after |> shouldBe __
    })

In [ ]:
koan "Every query is checked before it runs" (fun repo ->
    // repo.CheckQueries translates every query an assembly defines as a
    // value in its modules, runs none of them, and lists those Plasm cannot
    // translate. Call it in a test, or at start-up. Here it checks the
    // shop's modules (Shop.fs): Queries, whose queries translate, and
    // Mistakes, whose one query, Mistakes.shouting, does not.
    let problems = repo.CheckQueries(typeof<Customer>.Assembly, "Plasm.Koans.Shop")
    problems |> List.map (fun p -> p.Query) |> shouldBe __
    sqlSent().Length |> shouldBe __)

In [ ]:
koan "The model prints the tables it expects" (fun repo ->
    // repo.Ddl() prints the tables the model expects, with the constraint
    // names Plasm maps errors by. It runs nothing: it is the target for
    // the migration you write.
    let ddl = repo.Ddl()
    ddl.Contains "constraint customers_email_key unique (email)" |> shouldBe __
    ddl.Contains "constraint orders_customer_id_fkey foreign key (customer_id) references customers (id)" |> shouldBe __
    sqlSent().Length |> shouldBe __)

In [ ]:
koanTask "Transactions keep their meaning in a sandbox" (fun repo ->
    task {
        let failing =
            transaction {
                let! _ = Tx.insert (New.customer "Dan" "dan@example.com")
                return! Tx.fail "changed my mind"
            }

        let! _ = repo.Transact failing
        let! n = customers repo
        n |> shouldBe __
    })

In [ ]:
koanTask "A database error does not end a sandbox" (fun repo ->
    task {
        // Every koan already runs in a sandbox: repo is one.
        let! duplicate = repo.Transact(Tx.insert (New.customer "Ada again" "ada@example.com"): Tx<Customer, unit>)
        Result.isError duplicate |> shouldBe __
        let! later = repo.Transact addDan
        Result.isOk later |> shouldBe __
    })

In [ ]:
koanTask "The model can be checked against the database" (fun repo ->
    task {
        let! issues = repo.CheckSchema()
        issues |> shouldBe __
    })

In [ ]:
koanTask "Drift is reported precisely" (fun repo ->
    task {
        // DDL is transactional in PostgreSQL, so the sandbox undoes this too.
        let drop: Tx<int, unit> = Tx.executeSql $"alter table orders drop column cancellation_reason"
        let! _ = repo.Transact drop
        let! issues = repo.CheckSchema()
        let issue = (ok issues).Head
        (issue.Kind, issue.Model, issue.Field) |> shouldBe __
    })

## About unions with fields

A union whose cases carry facts is stored across columns and queried as F#

In [ ]:
open Plasm
open Plasm.Koans
open Plasm.Koans.Shop

let deliveryOf (order: int64) =
    plasm { for d in tableOf<Delivery> do where (d.OrderId = OrderId order) }

let orderIds (q: Query<Delivery>) (repo: Repo) =
    task {
        let! rows = repo.All q
        return ok rows |> List.map (fun d -> d.OrderId) |> List.sort
    }

In [ ]:
koanTask "A union with fields is read back whole" (fun repo ->
    task {
        // A Delivery's Stage is a union whose cases carry facts. Plasm
        // stores it across columns: the case's name in stage, and each
        // case's field in stage_<field>. Reading gives the union back.
        let! d = repo.One(deliveryOf 3L)
        (ok d).Value.Stage |> shouldBe __
    })

In [ ]:
koanTask "A case test is a query" (fun repo ->
    task {
        // Matching on a case becomes `stage = 'delivered'`.
        let delivered =
            plasm {
                for d in tableOf<Delivery> do
                    where (
                        match d.Stage with
                        | Delivered _ -> true
                        | _ -> false
                    )
            }

        let! ids = orderIds delivered repo
        ids |> shouldBe __
    })

In [ ]:
koanTask "Matching on a case's field" (fun repo ->
    task {
        // A case's field is its own column, so a match can test it.
        let bySwift =
            plasm {
                for d in tableOf<Delivery> do
                    where (
                        match d.Stage with
                        | InTransit carrier -> carrier = "Swift"
                        | _ -> false
                    )
            }

        let! ids = orderIds bySwift repo
        ids |> shouldBe __
    })

In [ ]:
koanTask "The domain's own member in a query" (fun repo ->
    task {
        // Delivery.Carrier is written in the domain, marked
        // [<ReflectedDefinition>]. Plasm reads its definition and
        // translates it, so the query says what the domain says.
        let carried = plasm { for d in tableOf<Delivery> do where (d.Carrier = Some "Swift") }
        let! ids = orderIds carried repo
        ids |> shouldBe __
    })

In [ ]:
koanTask "Comparing with a whole value" (fun repo ->
    task {
        // Comparing with a case compares the case and each of its fields.
        let signedByAda = plasm { for d in tableOf<Delivery> do where (d.Stage = Delivered "Ada") }
        let! ids = orderIds signedByAda repo
        ids |> shouldBe __
    })

In [ ]:
koanTask "Counting by stage" (fun repo ->
    task {
        // StageName is the domain's member too; grouping by it groups by case.
        let byStage =
            plasm {
                for d in tableOf<Delivery> do
                    groupBy d.StageName
                    select (d.StageName, Agg.count ())
            }

        let! rows = repo.All byStage
        ok rows |> List.sort |> shouldBe __
    })

In [ ]:
koanTask "Changing the stage clears the old stage's columns" (fun repo ->
    task {
        let! read = repo.One(deliveryOf 3L)
        let d = (ok read).Value
        let! r = repo.Transact(Tx.update (d, { d with Stage = Delivered "Bob" }): Tx<Delivery, unit>)
        ok r |> ignore
        // The carrier belonged to InTransit; Delivered has no carrier.
        let! carrier = repo.Sql<string option>($"select stage_carrier from deliveries where order_id = {3L}")
        ok carrier |> shouldBe __
    })

In [ ]:
koanTask "A row that is no case is refused" (fun repo ->
    task {
        // A stored stage that is none of the cases cannot become a
        // DeliveryStage, so reading it is an error naming the field.
        let lose: Tx<int, unit> = Tx.executeSql ($"update deliveries set stage = 'lost' where order_id = {2L}")
        let! _ = repo.Transact lose
        let! rows = repo.All(Query.from (tableOf<Delivery>))

        let refused =
            match rows with
            | Error e -> (string e).Contains "Stage"
            | Ok _ -> false

        refused |> shouldBe __
    })